# V13 — DINOv3 ViT-7B beku, **resampling model-native** (satu perubahan: bilinear → bicubic)

**Status:** notebook siap jalan. Belum pernah dieksekusi oleh siapa pun. Semua angka di bawah adalah angka historis dari `results/`, bukan hasil run ini.

---

## 1. Hipotesis

Champion saat ini (V10 `timm_control`) memakai resize **bilinear** langsung-kotak 256×256. Repo timm untuk `vit_7b_patch16_dinov3.lvd1689m` me-*resolve* interpolasi **bicubic** dengan `crop_pct = 1.0`. Deviasi ini **disengaja** di V7/V10 supaya perbandingan hanya mengisolasi representasi — dan V10 mencatatnya sendiri sebagai ablasi yang belum dikerjakan:

> `data_contract.json` → `preprocess_config.documented_future_ablation`:
> *"model-native bicubic preprocessing, not implemented as a candidate here"*

**Mengapa ini bukan detail kosmetik untuk masalah ini:**

1. 25.256 dari 26.527 citra train berarea < 65.536 px, artinya **95,2% citra di-*upsample*** ke 256×256. Pada *upsampling*, `antialias` tidak berpengaruh sama sekali, sehingga selisih bilinear vs bicubic **murni** selisih kernel: bilinear memberi hasil kabur (low-pass), bicubic mempertahankan frekuensi tinggi.
2. Seluruh 241 error R/O pada V7 berada di kelompok resolusi rendah tersebut (CONTEXT §7.4).
3. Batas Recyclable ↔ Organic adalah keputusan **tekstur**: kertas kusut vs daun kering, kilap plastik vs kulit buah. Tekstur adalah persis yang dihancurkan kernel resize yang salah.
4. DINOv3 dilatih dengan bicubic. Memakai bilinear saat inferensi adalah *mismatch* prapemrosesan terhadap pretraining backbone — bukan pilihan netral.

**Prediksi yang bisa gagal:** kalau hipotesis ini benar, perbaikan harus muncul terkonsentrasi di error R/O dan pada subset citra yang di-*upsample*. Kalau Macro F1 naik tapi error R/O tidak turun, hipotesisnya salah walaupun angkanya bagus — dan gerbang di §3 memang menolak kasus itu.

---

## 2. Satu perubahan, dan hanya satu

| Komponen | V10 champion | V13 | Berubah? |
|---|---|---|---|
| Backbone | `timm/vit_7b_patch16_dinov3.lvd1689m`, FP16, 100% beku | sama | tidak |
| Sharding | `explicit_contiguous_two_gpu_streamed` | sama | tidak |
| Resolusi | 256×256 langsung-kotak | sama | tidak |
| Antialias | `True` | sama | tidak |
| Mean/std | ImageNet | sama | tidak |
| **Interpolasi** | **bilinear** | **bicubic** | **YA** |
| Readout | final CLS ‖ mean patch final (8192-D) | sama | tidak |
| Head | LN → Linear(512) → GELU → Dropout(0.4) → Linear(3) | sama | tidak |
| Optimizer / LR / WD / loss / epoch / batch | AdamW 8e-4 / 1e-4 / weighted CE + LS 0.05 / 15 / 256 | sama | tidak |
| Fold | `StratifiedKFold(5, shuffle, seed 42)` | sama | tidak |
| Seed | 42 | sama | tidak |

Notebook menjalankan **dua varian pada backbone, sesi, kode, fold, dan seed yang sama**:

- `bilinear_v10_control` — replikasi persis resep champion.
- `bicubic_model_native` — hanya kernel resize yang berbeda.

Ini memperbaiki cacat perbandingan yang diakui di CONTEXT §13.5 (V7 vs V10 bukan *protocol-matched*): di sini kontrol dan treatment lahir dari satu proses yang sama, jadi selisihnya kausal terhadap kernel resize dan bukan terhadap backend, sesi, atau versi library.

---

## 3. Gerbang adopsi — dikunci sebelum ada satu pun hasil

`bicubic_model_native` diadopsi **hanya bila ketujuh syarat terpenuhi**:

1. Gain full 5-fold OOF Macro F1 atas kontrol **in-session** ≥ **+0.0003**.
2. Total error OOF **turun**.
3. Error R/O OOF **turun**.
4. Menang pada **≥ 3 dari 4** fold penilaian (fold 1–4).
5. F1 Electronic tidak turun lebih dari **0.0005**.
6. Gain tidak terkonsentrasi: sumbangan satu fold ≤ **75%** dari total gain positif.
7. Tidak ada akses test selama seleksi.

Tambahan stabilitas: tanda gain harus **konsisten pada seed 42, 314, dan 2026**. Seed 42 adalah satu-satunya yang dipakai untuk keputusan; 314 dan 2026 hanya konfirmasi, bukan seleksi.

**Gerbang integritas pipeline (dicek lebih dulu):** kontrol in-session harus mereproduksi angka V10 (OOF Macro F1 `0.9901662323927614`) dalam toleransi `0.0010`. Kalau gagal, ada yang rusak di pipeline dan hasil bicubic **tidak boleh dipercaya** — notebook tetap melanjutkan tetapi menandai seluruh keputusan sebagai tidak valid dan menolak adopsi.

Kalau bicubic ditolak, submission tetap dibuat — dari **kontrol**, yaitu resep V10 yang sudah menjadi champion. Jadi apa pun hasilnya, notebook berakhir dengan tepat **satu model, satu forward pass, satu argmax**, dan satu CSV yang tervalidasi.

---

## 4. Kontrak yang tetap dipatuhi

- Data uji hanya dibuka setelah `selection_decision.json` ditulis. Tidak pernah untuk training, validasi, seleksi model, atau tuning.
- *Single-model contract*: tanpa fold/seed ensembling, blending, TTA, multi-crop, hflip, model soup, specialist head, koreksi manual prediksi.
- Label validasi selalu label resmi asli. Tidak ada baris yang di-*drop* atau di-*relabel*.
- Dimensi citra sumber dicatat **hanya untuk analisis error**, tidak pernah masuk ke model sebagai fitur.
- Tanpa `device_map` otomatis, tanpa `DataParallel`, tanpa CPU/disk offload. Audit gagal-tertutup.

---

## 5. Perkiraan runtime (T4 × 2)

| Tahap | Perkiraan |
|---|---|
| Unduh + dispatch checkpoint | 15–30 menit |
| Ekstraksi train varian 1 (bilinear) | ± 1,9 jam |
| Ekstraksi train varian 2 (bicubic) | ± 1,9 jam |
| 2 × 5 fold head + 2 × 2 seed stabilitas | ± 15 menit |
| Ekstraksi test + inferensi + submission | ± 10 menit |
| **Total** | **± 4,5 jam** |

Cache fitur bersifat *resumable* dan atomik: sesi yang putus bisa dilanjutkan tanpa mengulang ekstraksi yang sudah valid."

## 6. Instruksi runtime — baca sebelum `Run All`

1. **Accelerator: `GPU T4 x2`.** Notebook menolak berjalan pada konfigurasi lain.
2. **Internet: ON.** Checkpoint DINOv3 bersifat *gated* dan diunduh dari Hugging Face.
3. **Kaggle Secret `HF_TOKEN`** harus ada (Add-ons → Secrets), dan akun yang sama sudah menyetujui lisensi di `https://huggingface.co/timm/vit_7b_patch16_dinov3.lvd1689m`.
4. **Dataset resmi terpasang**, sehingga ada tepat satu folder `train` berisi `0_Recyclable`, `1_Electronic`, `2_Organic` dengan jumlah 9.999 / 3.961 / 12.567, satu folder `test` berisi 1.458 citra, dan satu `submission.csv` resmi 1.458 baris.
5. **Jangan lampirkan CSV berlabel-manual test.** Ada penjaga yang menghentikan notebook bila menemukannya.
6. **Isi `TEAM_NAME`** di sel konfigurasi. Kalau masih `CHANGE_ME`, notebook tetap jalan tetapi mencetak peringatan besar dan nama berkas submission ikut salah.
7. **Mulai dari sesi bersih** supaya kedua GPU kosong saat checkpoint dimuat.

Sesudah selesai, sel terakhir mencetak satu blok ringkasan berisi semua angka yang perlu dilaporkan balik. Salin blok itu apa adanya.

In [ ]:
import os

# Set before torch is imported. Fragmentation guard only, not a correctness fix.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import gc
import hashlib
import json
import math
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from torchvision.transforms.functional import InterpolationMode
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# FILL THIS IN BEFORE SUBMITTING
# ---------------------------------------------------------------------------
TEAM_NAME = 'CHANGE_ME'

# ---------------------------------------------------------------------------
# EXPERIMENT IDENTITY
# ---------------------------------------------------------------------------
EXPERIMENT_NAME = 'v13_dinov3_vit7b_model_native_resampling'
REVISION_MARKER = 'v13_native_resample_2branch_v1'
EXPERIMENT_PURPOSE = (
    'isolate the resize interpolation kernel of the frozen DINOv3 ViT-7B readout; '
    'one change against the V10 champion recipe, control and treatment in the same session'
)

# ---------------------------------------------------------------------------
# BACKBONE CONTRACT (identical to V10)
# ---------------------------------------------------------------------------
TIMM_MODEL_NAME = 'vit_7b_patch16_dinov3.lvd1689m'
TIMM_HF_REPO_ID = 'timm/vit_7b_patch16_dinov3.lvd1689m'
TRANSFORMERS_MODEL_ID = 'facebook/dinov3-vit7b16-pretrain-lvd1689m'
MODEL_REVISION_REQUEST = 'main'

IMAGE_SIZE = 256
PATCH_SIZE_EXPECTED = 16
GRID_SIZE = 16
EXPECTED_PATCH_TOKENS = 256
EXPECTED_PREFIX_TOKENS = 5
EXPECTED_REGISTER_TOKENS = 4
EXPECTED_SEQUENCE_TOKENS = 261
EXPECTED_HIDDEN_SIZE = 4096

BACKBONE_FROZEN = True
BACKBONE_DTYPE = torch.float16
BACKBONE_DTYPE_NAME = 'float16'
POOLING_COMPUTE_DTYPE = torch.float32
POOLING_COMPUTE_DTYPE_NAME = 'float32'
FEATURE_STORAGE_DTYPE = torch.float16
FEATURE_STORAGE_DTYPE_NAME = 'float16'

SHARDING_STRATEGY = 'explicit_contiguous_two_gpu_streamed'
GPU_STATIC_MEMORY_LIMIT_GIB = 10.5
GPU_MIN_RUNTIME_HEADROOM_GIB = 3.0
CHECKPOINT_MIN_TOTAL_GIB = 1.0
CHECKPOINT_MAX_TOTAL_GIB = 60.0
OUTPUT_SIDE_MODULE_NAMES = ('norm', 'fc_norm', 'head_norm', 'attn_pool', 'head_drop', 'head', 'head_dist')

FEATURE_BATCH_SIZE = 4
FEATURE_LOADER_WORKERS = 2
FEATURE_SHARD_SIZE = 256
FEATURE_MAX_ABS_LIMIT = 1e6
MIN_EXTRACTION_IMAGES_PER_SECOND = 0.8
BENCHMARK_WARMUP_IMAGES = 8
BENCHMARK_TIMED_IMAGES = 24

# ---------------------------------------------------------------------------
# READOUT CONTRACT
# Only the two branches the champion actually consumes. The 17-branch rich readout
# was screened and rejected in V10, so it is deliberately not rebuilt here.
# ---------------------------------------------------------------------------
BRANCH_NAMES = ('final_cls', 'final_patch_mean')
NUM_BRANCHES = len(BRANCH_NAMES)
READOUT_DESCRIPTION = (
    'timm forward_features applies the official pretrained DINOv3 final norm exactly once; '
    'branch 0 is prefix token 0 (CLS) and branch 1 is the mean over the 256 true patch tokens '
    'with all 4 register tokens excluded'
)

# ---------------------------------------------------------------------------
# THE SINGLE EXPERIMENTAL VARIABLE
# ---------------------------------------------------------------------------
CONTROL_VARIANT = 'bilinear_v10_control'
NATIVE_VARIANT = 'bicubic_model_native'
VARIANT_ORDER = (CONTROL_VARIANT, NATIVE_VARIANT)
VARIANT_INTERPOLATION = {CONTROL_VARIANT: 'bilinear', NATIVE_VARIANT: 'bicubic'}
INTERPOLATION_MODES = {'bilinear': InterpolationMode.BILINEAR, 'bicubic': InterpolationMode.BICUBIC}

# ---------------------------------------------------------------------------
# OFFICIAL DATA CONTRACT
# ---------------------------------------------------------------------------
CLASS_DIR_TO_LABEL = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_RAW_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_TRAIN_ROWS = 26527
EXPECTED_TEST_ROWS = 1458
SUBMISSION_COLUMNS = ['id', 'predicted']
ALLOWED_IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
LARGE_IMAGE_MP_THRESHOLD = 8_000_000
LARGE_IMAGE_MAX_SOURCE_SIDE = 4096
SAFE_DECODE_MAX_SIDE = 1024
PIL_MAX_IMAGE_PIXELS = 200_000_000

# ---------------------------------------------------------------------------
# HEAD PROTOCOL (identical to V10, identical for both variants)
# ---------------------------------------------------------------------------
SEED = 42
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
ASSESSMENT_FOLDS = (1, 2, 3, 4)
STABILITY_SEEDS = (314, 2026)
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
HEAD_LEARNING_RATE = 8e-4
HEAD_WEIGHT_DECAY = 1e-4
HEAD_BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 512
GRAD_CLIP_MAX_NORM = 1.0
LABEL_SMOOTHING = 0.05
HEAD_HIDDEN_DIM = 512
HEAD_DROPOUT = 0.40

# ---------------------------------------------------------------------------
# LOCKED ADOPTION GATES (fixed before any V13 result exists)
# ---------------------------------------------------------------------------
ADOPTION_MIN_OOF_MACRO_F1_GAIN = 0.0003
ADOPTION_MIN_ASSESSMENT_FOLDS_WON = 3
ADOPTION_MAX_ELECTRONIC_F1_DROP = 0.0005
ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE = 0.75
STRONG_EVIDENCE_MACRO_F1_GAIN = 0.0015
CONTROL_REPRODUCTION_TOLERANCE = 0.0010
RELOAD_TOLERANCE = 1e-5

# ---------------------------------------------------------------------------
# MEASURED HISTORICAL FACTS (from results/, never a claim about this run)
# ---------------------------------------------------------------------------
V10_REFERENCE = {
    'source': 'results/v10_dinov3_vit7b_timm_rich_spatial/selection_decision.json',
    'candidate': 'timm_control',
    'fold_assignment_sha256': '0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c',
    'labels_sha256': '719d36e9918b1fd09711c06d1b259fca08ca6eb4d973de1388d5e3ddecf22d35',
    'ordered_image_ids_sha256': '210f8f14797be49bad962e81f1da660a257ec28ee2136b54a7516950a66cb2e8',
    'oof_macro_f1': 0.9901662323927614,
    'oof_total_errors': 287,
    'oof_ro_errors': 242,
    'oof_electronic_errors': 16,
    'oof_f1_recyclable': 0.9863779252532309,
    'oof_f1_electronic': 0.994328922495274,
    'oof_f1_organic': 0.989791849429779,
    'oof_recyclable_to_organic': 96,
    'oof_organic_to_recyclable': 146,
    'fold0_macro_f1': 0.9897706675032462,
    'full_training_epochs': 5,
    'test_predicted_class_distribution': {'0': 520, '1': 201, '2': 737},
    'note': 'same folds, same labels, same head recipe; produced by the timm backend with bilinear resize',
}

# ---------------------------------------------------------------------------
# CONTRACT ASSERTIONS
# ---------------------------------------------------------------------------
assert IMAGE_SIZE == 256 and PATCH_SIZE_EXPECTED == 16 and GRID_SIZE == 16
assert EXPECTED_PATCH_TOKENS == GRID_SIZE * GRID_SIZE
assert EXPECTED_SEQUENCE_TOKENS == EXPECTED_PREFIX_TOKENS + EXPECTED_PATCH_TOKENS
assert EXPECTED_PREFIX_TOKENS == EXPECTED_REGISTER_TOKENS + 1
assert BACKBONE_FROZEN and BACKBONE_DTYPE is torch.float16
assert POOLING_COMPUTE_DTYPE is torch.float32 and FEATURE_STORAGE_DTYPE is torch.float16
assert SEED == 42 and N_FOLDS == 5 and DEVELOPMENT_FOLD == 0 and ASSESSMENT_FOLDS == (1, 2, 3, 4)
assert sum(EXPECTED_RAW_COUNTS.values()) == EXPECTED_TRAIN_ROWS
assert set(VARIANT_ORDER) == set(VARIANT_INTERPOLATION)
assert VARIANT_ORDER[0] == CONTROL_VARIANT, 'the control must run first so it can validate the pipeline'
assert len(set(VARIANT_INTERPOLATION.values())) == 2, 'the two variants must differ in exactly the kernel'
assert NUM_BRANCHES == 2 and BRANCH_NAMES == ('final_cls', 'final_patch_mean')
assert SEED not in STABILITY_SEEDS

# ---------------------------------------------------------------------------
# TEST-ISOLATION STATE (flipped only inside the post-lock test section)
# ---------------------------------------------------------------------------
TEST_ACCESS_STATE = {'test_images_accessed': False, 'manual_test_labels_accessed': False}
NOTEBOOK_STARTED_AT = time.perf_counter()

print('Experiment :', EXPERIMENT_NAME)
print('Variants   :')
for variant_name in VARIANT_ORDER:
    print('  {:<22} interpolation = {}'.format(variant_name, VARIANT_INTERPOLATION[variant_name]))
print('Branches   :', BRANCH_NAMES)
print('Adoption gate: OOF macro F1 gain >= {:+.4f}, >= {} of 4 assessment folds won'.format(
    ADOPTION_MIN_OOF_MACRO_F1_GAIN, ADOPTION_MIN_ASSESSMENT_FOLDS_WON
))
if TEAM_NAME == 'CHANGE_ME':
    print()
    print('!' * 78)
    print('!! TEAM_NAME is still "CHANGE_ME". Set it before you submit anything.        !!')
    print('!' * 78)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')
Image.MAX_IMAGE_PIXELS = PIL_MAX_IMAGE_PIXELS
ImageFile.LOAD_TRUNCATED_IMAGES = False

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError(
        'This notebook requires a Kaggle runtime with /kaggle/working. '
        'Run it on Kaggle with the GPU T4 x2 accelerator.'
    )
KAGGLE_INPUT = Path('/kaggle/input')
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
CACHE_DIR = ARTIFACT_DIR / 'feature_cache'
HEADS_DIR = ARTIFACT_DIR / 'heads'
FIGURES_DIR = ARTIFACT_DIR / 'figures'
for directory in (ARTIFACT_DIR, CACHE_DIR, HEADS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SELECTION_DECISION_PATH = ARTIFACT_DIR / 'selection_decision.json'
PREFLIGHT_REPORT_PATH = ARTIFACT_DIR / 'preflight_report.json'


def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError('Not JSON serializable: {}'.format(type(value).__name__))


def sha256_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, default=json_default)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def write_npy_atomic(array, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp.npy')
    np.save(temporary, array)
    os.replace(temporary, path)


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def host_rss_gib():
    resident_pages = int(Path('/proc/self/statm').read_text().split()[1])
    return resident_pages * os.sysconf('SC_PAGE_SIZE') / 2 ** 30


def gpu_memory_rows():
    rows = []
    for device_id in range(torch.cuda.device_count()):
        rows.append({
            'device_id': device_id,
            'allocated_gib': torch.cuda.memory_allocated(device_id) / 2 ** 30,
            'reserved_gib': torch.cuda.memory_reserved(device_id) / 2 ** 30,
            'peak_allocated_gib': torch.cuda.max_memory_allocated(device_id) / 2 ** 30,
        })
    return rows


def cuda_memory_snapshot(label):
    rows = []
    for device_id in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        rows.append({
            'label': label,
            'device_id': device_id,
            'free_gib': free_bytes / 2 ** 30,
            'total_gib': total_bytes / 2 ** 30,
            'allocated_gib': torch.cuda.memory_allocated(device_id) / 2 ** 30,
            'reserved_gib': torch.cuda.memory_reserved(device_id) / 2 ** 30,
        })
    return rows


def release_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sanitize_for_filename(value):
    cleaned = ''.join(character if character.isalnum() or character in {'-', '_'} else '_' for character in str(value))
    return cleaned.strip('_') or 'UNSET'


def elapsed_minutes():
    return (time.perf_counter() - NOTEBOOK_STARTED_AT) / 60.0


print('Artifact directory:', ARTIFACT_DIR)
print('Seed locked at    :', SEED)

In [ ]:
import accelerate
import safetensors
import timm
import timm.data  # explicit: timm/__init__.py does not import the data subpackage

ENVIRONMENT_INFO = {
    'python_version': sys.version,
    'platform': platform.platform(),
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
    'cudnn_version': torch.backends.cudnn.version(),
    'timm_version': timm.__version__,
    'accelerate_version': accelerate.__version__,
    'safetensors_version': safetensors.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
}

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError(
        'V13 requires exactly two visible CUDA devices, found {}. '
        'Set the Kaggle accelerator to "GPU T4 x2".'.format(
            torch.cuda.device_count() if torch.cuda.is_available() else 0
        )
    )
GPU_RUNTIME_INFO = []
for device_id in (0, 1):
    properties = torch.cuda.get_device_properties(device_id)
    upper_name = properties.name.upper()
    if 'TESLA T4' not in upper_name and upper_name.strip() != 'T4':
        raise RuntimeError(
            'cuda:{} is "{}", not an NVIDIA Tesla T4. The locked protocol requires two Tesla T4 GPUs.'.format(
                device_id, properties.name
            )
        )
    GPU_RUNTIME_INFO.append({
        'device_id': device_id,
        'name': properties.name,
        'total_vram_gib': properties.total_memory / 2 ** 30,
        'multi_processor_count': properties.multi_processor_count,
    })

if not KAGGLE_INPUT.is_dir():
    raise RuntimeError('Attach the competition dataset: /kaggle/input was not found.')

blocked_label_files = []
for candidate_path in KAGGLE_INPUT.rglob('*.csv'):
    normalized = candidate_path.name.lower().replace('_', '-')
    if 'manual' in normalized and 'label' in normalized:
        blocked_label_files.append(str(candidate_path))
if blocked_label_files:
    raise RuntimeError(
        'Competition safety guard triggered: manual test-label CSV files are attached as notebook inputs. '
        'Detach them before running V13:' + '\n' + '\n'.join(blocked_label_files)
    )

LOCKED_CONTRACT = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'experiment_purpose': EXPERIMENT_PURPOSE,
    'backbone_backend': 'timm',
    'timm_model_name': TIMM_MODEL_NAME,
    'timm_hf_repo_id': TIMM_HF_REPO_ID,
    'transformers_model_id': TRANSFORMERS_MODEL_ID,
    'backbone_frozen': BACKBONE_FROZEN,
    'backbone_dtype': BACKBONE_DTYPE_NAME,
    'pooling_compute_dtype': POOLING_COMPUTE_DTYPE_NAME,
    'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
    'image_size': IMAGE_SIZE,
    'patch_size_expected': PATCH_SIZE_EXPECTED,
    'token_contract': {
        'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
        'prefix_token_count': EXPECTED_PREFIX_TOKENS,
        'register_token_count': EXPECTED_REGISTER_TOKENS,
        'patch_token_count': EXPECTED_PATCH_TOKENS,
        'hidden_size': EXPECTED_HIDDEN_SIZE,
        'cls_position': 'prefix token 0',
        'register_tokens_excluded_from_every_patch_statistic': True,
    },
    'branch_names': list(BRANCH_NAMES),
    'readout': READOUT_DESCRIPTION,
    'sharding_strategy': SHARDING_STRATEGY,
    'experimental_variable': {
        'name': 'resize_interpolation_kernel',
        'variants': {name: VARIANT_INTERPOLATION[name] for name in VARIANT_ORDER},
        'everything_else_held_fixed': True,
        'geometry': 'direct_square 256x256, antialias True, no center crop, no letterbox, no augmentation',
    },
    'seed': SEED,
    'stability_seeds': list(STABILITY_SEEDS),
    'n_folds': N_FOLDS,
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'head_protocol': {
        'architecture': 'LayerNorm(8192) -> Linear({}) -> GELU -> Dropout({}) -> Linear({})'.format(
            HEAD_HIDDEN_DIM, HEAD_DROPOUT, NUM_CLASSES
        ),
        'max_epochs': MAX_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'batch_size': HEAD_BATCH_SIZE,
        'predict_batch_size': PREDICT_BATCH_SIZE,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
        'label_smoothing': LABEL_SMOOTHING,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'class_weights': 'total / (num_classes * class_count) from the current training partition',
        'model_selection_metric': 'validation Macro F1, log loss as secondary diagnostic',
        'threshold_tuning': False,
    },
    'gates': {
        'adoption_min_oof_macro_f1_gain': ADOPTION_MIN_OOF_MACRO_F1_GAIN,
        'adoption_min_assessment_folds_won': ADOPTION_MIN_ASSESSMENT_FOLDS_WON,
        'adoption_max_electronic_f1_drop': ADOPTION_MAX_ELECTRONIC_F1_DROP,
        'adoption_max_single_fold_gain_share': ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE,
        'strong_evidence_macro_f1_gain': STRONG_EVIDENCE_MACRO_F1_GAIN,
        'control_reproduction_tolerance': CONTROL_REPRODUCTION_TOLERANCE,
        'seed_sign_consistency_required': True,
    },
    'single_model_contract': {
        'fold_ensembling': False,
        'seed_ensembling': False,
        'probability_blending': False,
        'model_soup': False,
        'specialist_heads': False,
        'test_time_augmentation': False,
        'multi_crop_inference': False,
        'horizontal_flip_inference': False,
        'multiple_backbone_families': False,
        'manual_prediction_correction': False,
    },
    'official_raw_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
    'official_test_rows': EXPECTED_TEST_ROWS,
    'historical_reference_v10': V10_REFERENCE,
}
LOCKED_CONTRACT_SIGNATURE = sha256_json(LOCKED_CONTRACT)

CONFIG_SNAPSHOT = {
    'team_name': TEAM_NAME,
    'artifact_dir': str(ARTIFACT_DIR),
    'feature_batch_size': FEATURE_BATCH_SIZE,
    'feature_loader_workers': FEATURE_LOADER_WORKERS,
    'feature_shard_size': FEATURE_SHARD_SIZE,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'test_access_state_at_configuration_time': dict(TEST_ACCESS_STATE),
}

write_json_atomic({'environment': ENVIRONMENT_INFO, 'gpus': GPU_RUNTIME_INFO}, ARTIFACT_DIR / 'environment.json')
write_json_atomic(CONFIG_SNAPSHOT, ARTIFACT_DIR / 'config.json')
write_json_atomic(
    {'locked_contract': LOCKED_CONTRACT, 'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE},
    ARTIFACT_DIR / 'locked_contract.json',
)

print(json.dumps(ENVIRONMENT_INFO, indent=2))
for info in GPU_RUNTIME_INFO:
    print('cuda:{device_id}: {name}, {total_vram_gib:.2f} GiB'.format(**info))
print('Manual test-label guard passed: no such CSV attached.')
print('Locked contract signature:', LOCKED_CONTRACT_SIGNATURE)

In [ ]:
from huggingface_hub import model_info, snapshot_download

HF_ACTION_HINT = (
    'Actions to resolve:' + '\n'
    '  1. Create a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) holding a Hugging Face read token.' + '\n'
    '  2. Open https://huggingface.co/{} with that same account and accept the DINOv3 license.' + '\n'
    '  3. Enable notebook internet access.' + '\n'
    '  4. Do not downgrade to a smaller backbone; the locked protocol requires DINOv3 ViT-7B.'
)


def resolve_hf_token():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            'Missing Hugging Face token for the gated DINOv3 checkpoint.' + '\n'
            + HF_ACTION_HINT.format(TIMM_HF_REPO_ID)
        )
    return token


HF_TOKEN_VALUE = resolve_hf_token()
print('Hugging Face authentication present; the token is never printed or persisted.')

try:
    RESOLVED_REVISION = model_info(TIMM_HF_REPO_ID, revision=MODEL_REVISION_REQUEST, token=HF_TOKEN_VALUE).sha
except Exception as exc:
    raise RuntimeError(
        'Could not resolve an immutable revision for {}. This usually means the license was not accepted, '
        'the token lacks access, or the notebook has no internet.'.format(TIMM_HF_REPO_ID) + '\n'
        + HF_ACTION_HINT.format(TIMM_HF_REPO_ID)
    ) from exc
if not RESOLVED_REVISION or len(RESOLVED_REVISION) != 40 or any(
    character not in '0123456789abcdef' for character in RESOLVED_REVISION.lower()
):
    raise RuntimeError(
        'The Hub did not return an immutable 40-character commit SHA for {} (got {!r}).'.format(
            TIMM_HF_REPO_ID, RESOLVED_REVISION
        )
    )

MODEL_DOWNLOAD_STARTED_AT = time.perf_counter()
try:
    MODEL_SNAPSHOT_DIR = snapshot_download(
        repo_id=TIMM_HF_REPO_ID,
        revision=RESOLVED_REVISION,
        token=HF_TOKEN_VALUE,
        allow_patterns=['*.safetensors', '*.safetensors.index.json', '*.json'],
    )
except Exception as exc:
    raise RuntimeError(
        'Failed to download {} at revision {}. Check disk space (the checkpoint is about 25 GB), '
        'internet access, and license acceptance.'.format(TIMM_HF_REPO_ID, RESOLVED_REVISION) + '\n'
        + HF_ACTION_HINT.format(TIMM_HF_REPO_ID)
    ) from exc
MODEL_DOWNLOAD_SECONDS = time.perf_counter() - MODEL_DOWNLOAD_STARTED_AT

snapshot_files = sorted(path.name for path in Path(MODEL_SNAPSHOT_DIR).iterdir() if path.is_file())
snapshot_bytes = sum(path.stat().st_size for path in Path(MODEL_SNAPSHOT_DIR).rglob('*') if path.is_file())
print('Repository        :', TIMM_HF_REPO_ID)
print('Immutable revision:', RESOLVED_REVISION)
print('V10 used revision : b40f6c827cb02b13c7a23a76cf598489bb0f94e5')
if RESOLVED_REVISION != 'b40f6c827cb02b13c7a23a76cf598489bb0f94e5':
    print('NOTE: the resolved revision differs from the one V10 used. Record this; it weakens the')
    print('      comparability of the in-session control against the recorded V10 numbers, but the')
    print('      control-vs-treatment comparison inside this run stays valid because both variants')
    print('      use this same revision.')
print('Snapshot files    :', snapshot_files)
print('Snapshot size     : {:.2f} GiB (downloaded in {:.1f} s)'.format(snapshot_bytes / 2 ** 30, MODEL_DOWNLOAD_SECONDS))

In [ ]:
def discover_named_dirs(base, wanted, max_depth=3, prune_names=()):
    if not base.is_dir():
        return []
    found = []
    stack = [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            lowered = child.name.lower()
            if lowered in prune_names:
                # Pruned without enumeration so official test images stay untouched during selection.
                continue
            if lowered == wanted:
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found


def exact_raw_train_signature(train_dir):
    if not train_dir.is_dir():
        return None
    try:
        root_entries = list(train_dir.iterdir())
        if {path.name for path in root_entries} != set(CLASS_DIR_TO_LABEL):
            return None
        if any(path.is_symlink() or not path.is_dir() for path in root_entries):
            return None
        counts = {}
        for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
            entries = list((train_dir / class_dir_name).iterdir())
            if any(
                entry.is_symlink() or not entry.is_file() or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
                for entry in entries
            ):
                return None
            counts[label] = len(entries)
        return counts if counts == EXPECTED_RAW_COUNTS else None
    except OSError:
        return None


def resolve_train_dir():
    candidates = [Path.cwd() / 'data' / 'train', KAGGLE_WORKING / 'data' / 'train']
    candidates.extend(discover_named_dirs(KAGGLE_INPUT, 'train', max_depth=3, prune_names={'test'}))
    matches = {
        candidate.resolve()
        for candidate in candidates
        if exact_raw_train_signature(candidate) == EXPECTED_RAW_COUNTS
    }
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(path) for path in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError(
            'Expected exactly one official raw train directory with class counts {}; found {}:'.format(
                EXPECTED_RAW_COUNTS, len(matches)
            ) + '\n' + rendered
        )
    return matches.pop()


def load_rgb_image(path):
    """Identical decode path to V7 and V10, including the decoder-memory guard."""
    try:
        with Image.open(path) as image:
            source_width, source_height = image.size
            if source_width <= 0 or source_height <= 0:
                raise ValueError('invalid image dimensions')
            needs_downscale = (
                source_width * source_height >= LARGE_IMAGE_MP_THRESHOLD
                or max(source_width, source_height) > LARGE_IMAGE_MAX_SOURCE_SIDE
            )
            image.draft('RGB', (SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE))
            rgb_image = ImageOps.exif_transpose(image).convert('RGB')
            if needs_downscale:
                rgb_image.thumbnail((SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE), Image.Resampling.LANCZOS)
            return rgb_image.copy()
    except Exception as exc:
        raise RuntimeError('Image cannot be decoded: {}'.format(path)) from exc


def build_transform(interpolation_name):
    """The ONLY thing that differs between the two variants is `interpolation`."""
    if interpolation_name not in INTERPOLATION_MODES:
        raise KeyError('Unknown interpolation: {}'.format(interpolation_name))
    return v2.Compose([
        v2.ToImage(),
        v2.Resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=INTERPOLATION_MODES[interpolation_name],
            antialias=True,
        ),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


VARIANT_TRANSFORMS = {name: build_transform(VARIANT_INTERPOLATION[name]) for name in VARIANT_ORDER}


def build_preprocess_config(variant_name):
    return {
        'variant': variant_name,
        'open': 'PIL, EXIF transpose, RGB conversion',
        'large_image_guard': {
            'megapixel_threshold': LARGE_IMAGE_MP_THRESHOLD,
            'max_source_side': LARGE_IMAGE_MAX_SOURCE_SIDE,
            'downscaled_to_max_side': SAFE_DECODE_MAX_SIDE,
            'resample': 'LANCZOS',
            'purpose': 'decoder memory safety only, never a predictive feature',
        },
        'resize_geometry': 'direct_square',
        'spatial_override': {'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
        'interpolation': VARIANT_INTERPOLATION[variant_name],
        'antialias': True,
        'rescale_factor': 1.0 / 255.0,
        'image_mean': list(IMAGENET_MEAN),
        'image_std': list(IMAGENET_STD),
        'center_crop': False,
        'letterbox': False,
        'augmentation': 'none',
        'feature_view': 'orig',
    }


VARIANT_PREPROCESS_CONFIG = {name: build_preprocess_config(name) for name in VARIANT_ORDER}
VARIANT_PREPROCESS_SIGNATURE = {name: sha256_json(VARIANT_PREPROCESS_CONFIG[name]) for name in VARIANT_ORDER}
if len(set(VARIANT_PREPROCESS_SIGNATURE.values())) != len(VARIANT_ORDER):
    raise RuntimeError('The two variants produced identical preprocess signatures; the caches would collide.')


class ImageRows(Dataset):
    """Deterministic, order-preserving image reader bound to one variant transform."""

    def __init__(self, paths, transform):
        self.paths = [str(path) for path in paths]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, position):
        return self.transform(load_rgb_image(self.paths[position]))


TRAIN_DIR = resolve_train_dir()
print('Official raw train root:', TRAIN_DIR)
for variant_name in VARIANT_ORDER:
    print('  {:<22} interpolation={:<8} preprocess_signature={}'.format(
        variant_name, VARIANT_INTERPOLATION[variant_name], VARIANT_PREPROCESS_SIGNATURE[variant_name]
    ))
print('V10 preprocess signature (bilinear): f9f83661728d270736b581bf88e5497c57ac07a885fbf55c377aa75c5bddd6c0')
print('NOTE: V13 signatures differ from V10 because the payload now records the variant name.')
print('      Compare the individual fields above, not the hash, when checking parity with V10.')

In [ ]:
def read_source_size(path):
    """Header-only read. Used for error stratification only, never as a model feature."""
    try:
        with Image.open(path) as image:
            width, height = image.size
            return int(width), int(height)
    except Exception:
        return 0, 0


def index_raw_train(train_dir):
    rows = []
    observed_counts = {}
    for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
        class_dir = train_dir / class_dir_name
        entries = sorted(class_dir.iterdir(), key=lambda path: path.name)
        invalid = [
            entry.name
            for entry in entries
            if entry.is_symlink() or not entry.is_file() or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
        ]
        if invalid:
            raise RuntimeError('Nested, unsupported or symlink entries in {}: {}'.format(class_dir, invalid[:10]))
        observed_counts[label] = len(entries)
        for path in entries:
            rows.append({
                'image_id': path.relative_to(train_dir).as_posix(),
                'label': label,
                'class_name': CLASS_NAMES[label],
                'path': str(path.resolve()),
                'file_size_bytes': path.stat().st_size,
            })
    if observed_counts != EXPECTED_RAW_COUNTS:
        raise RuntimeError(
            'Raw class counts do not match the official contract: observed={}, expected={}.'.format(
                observed_counts, EXPECTED_RAW_COUNTS
            )
        )
    frame = pd.DataFrame(rows).sort_values('image_id').reset_index(drop=True)
    if len(frame) != EXPECTED_TRAIN_ROWS or not frame['image_id'].is_unique:
        raise RuntimeError('Raw-train row count or image_id uniqueness failed.')
    return frame


train_df = index_raw_train(TRAIN_DIR)

# Fold reconstruction: byte-for-byte the V7/V10 definition.
train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold_index, (_, valid_positions) in enumerate(splitter.split(train_df, train_df['label'])):
    train_df.loc[valid_positions, 'fold'] = fold_index
train_df['fold'] = train_df['fold'].astype(np.int64)
if set(train_df['fold'].tolist()) != set(range(N_FOLDS)) or (train_df['fold'] < 0).any():
    raise RuntimeError('Five-fold assignment is incomplete.')
if train_df['label'].value_counts().sort_index().to_dict() != EXPECTED_RAW_COUNTS:
    raise RuntimeError('Full index no longer matches the official raw counts.')

ORDERED_IMAGE_IDS_SHA256 = sha256_json(train_df['image_id'].tolist())
LABELS_SHA256 = sha256_json(train_df['label'].astype(int).tolist())
FOLD_ASSIGNMENT_SHA256 = sha256_json(train_df[['image_id', 'label', 'fold']].to_dict('records'))
RAW_DATA_SHA256 = sha256_json(train_df[['image_id', 'label', 'file_size_bytes']].to_dict('records'))

# ---------------------------------------------------------------------------
# HARD PROTOCOL-MATCH GATE
# If any of these differ from V10, the comparison against the recorded champion
# numbers is meaningless and the run must not continue.
# ---------------------------------------------------------------------------
PROTOCOL_MATCH_CHECKS = {
    'ordered_image_ids_match_v10': ORDERED_IMAGE_IDS_SHA256 == V10_REFERENCE['ordered_image_ids_sha256'],
    'labels_match_v10': LABELS_SHA256 == V10_REFERENCE['labels_sha256'],
    'fold_assignment_matches_v10': FOLD_ASSIGNMENT_SHA256 == V10_REFERENCE['fold_assignment_sha256'],
}
failed_protocol_checks = [name for name, passed in PROTOCOL_MATCH_CHECKS.items() if not passed]
if failed_protocol_checks:
    raise RuntimeError(
        'Protocol match against V10 failed: {}. The dataset or the fold construction is not the one the '
        'champion was measured on, so no comparison in this notebook would be valid. '
        'Expected fold sha256 {}, got {}.'.format(
            failed_protocol_checks, V10_REFERENCE['fold_assignment_sha256'], FOLD_ASSIGNMENT_SHA256
        )
    )

# Source-size census. Analysis stratum only; explicitly not a model input.
print('Reading image headers for the resolution census (analysis only, never a feature)...')
census_started = time.perf_counter()
source_sizes = [read_source_size(path) for path in tqdm(train_df['path'].tolist(), desc='image headers')]
train_df['source_width'] = [size[0] for size in source_sizes]
train_df['source_height'] = [size[1] for size in source_sizes]
train_df['source_area'] = train_df['source_width'] * train_df['source_height']
train_df['is_upsampled'] = train_df['source_area'] < (IMAGE_SIZE * IMAGE_SIZE)
UPSAMPLED_ROWS = int(train_df['is_upsampled'].sum())
CENSUS_SECONDS = time.perf_counter() - census_started

FOLD_CLASS_COUNTS = train_df.groupby(['fold', 'label']).size().unstack(fill_value=0)

FOLD_SIGNATURE = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'fold_definition': {
        'implementation': 'sklearn.model_selection.StratifiedKFold',
        'n_splits': N_FOLDS,
        'shuffle': True,
        'random_state': SEED,
        'ordering': 'relative official image_id ascending before the split',
    },
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
    'fold_class_counts': {str(fold): row.to_dict() for fold, row in FOLD_CLASS_COUNTS.iterrows()},
    'validation_labels': 'always the official original labels; no drop and no relabel',
    'matches_v10_fold_signature': True,
}
DATA_CONTRACT = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'canonical_data': 'official raw labeled train images only',
    'raw_train_root': str(TRAIN_DIR),
    'official_class_codes': {str(index): name for index, name in enumerate(CLASS_NAMES)},
    'class_dir_to_label': CLASS_DIR_TO_LABEL,
    'raw_class_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
    'row_count': int(len(train_df)),
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SHA256,
    'labels_sha256': LABELS_SHA256,
    'raw_data_sha256': RAW_DATA_SHA256,
    'fold_signature': FOLD_SIGNATURE,
    'preprocess_config_by_variant': VARIANT_PREPROCESS_CONFIG,
    'preprocess_signature_by_variant': VARIANT_PREPROCESS_SIGNATURE,
    'external_labeled_images_used': False,
    'metadata_used_as_feature': False,
    'source_dimensions_recorded_for': 'error stratification and the upsampling census only',
    'dropped_or_relabeled_rows': 0,
    'upsampled_row_count': UPSAMPLED_ROWS,
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
DATA_CONTRACT_SIGNATURE = sha256_json(DATA_CONTRACT)

write_json_atomic(FOLD_SIGNATURE, ARTIFACT_DIR / 'fold_signature.json')
write_json_atomic(
    {'data_contract': DATA_CONTRACT, 'data_contract_signature': DATA_CONTRACT_SIGNATURE},
    ARTIFACT_DIR / 'data_contract.json',
)
write_csv_atomic(
    train_df[['image_id', 'label', 'class_name', 'fold', 'source_width', 'source_height', 'is_upsampled']],
    ARTIFACT_DIR / 'fold_assignments.csv',
)

cached_labels = torch.tensor(train_df['label'].to_numpy(), dtype=torch.long)
cached_folds = torch.tensor(train_df['fold'].to_numpy(), dtype=torch.long)

print()
print('Rows                    :', len(train_df))
print('Class counts            :', train_df['label'].value_counts().sort_index().to_dict())
print('Fold assignment sha256  :', FOLD_ASSIGNMENT_SHA256)
print('  -> matches V10        : True (hard gate passed)')
print('Upsampled to 256x256    : {} of {} rows ({:.1f}%)  [CONTEXT 7.4 expects 25256]'.format(
    UPSAMPLED_ROWS, len(train_df), 100.0 * UPSAMPLED_ROWS / len(train_df)
))
print('Census took             : {:.1f} s'.format(CENSUS_SECONDS))
print('Data contract signature :', DATA_CONTRACT_SIGNATURE)

In [ ]:
from accelerate import dispatch_model, init_empty_weights
from accelerate.utils import set_module_tensor_to_device
from safetensors import safe_open

DISPATCH_FAILURE_HINT = (
    'The DINOv3 ViT-7B backbone could not be placed across the two Tesla T4 GPUs.' + '\n'
    'This is a weight-loading failure that happens before any image is processed, so changing' + '\n'
    'FEATURE_BATCH_SIZE, DataLoader workers or the feature shard size will NOT help.' + '\n'
    'Checks to run, in order:' + '\n'
    '  1. Confirm the Kaggle accelerator is "GPU T4 x2" and no other process holds GPU memory.' + '\n'
    '  2. Restart the session so both GPUs start empty, then rerun from the top.' + '\n'
    '  3. Inspect planned_device_map.json and model_load_failure_report.json in the artifact directory.' + '\n'
    'CPU offload, disk offload, single-GPU placement and smaller backbones are all forbidden.'
)

DTYPE_ELEMENT_BYTES = torch.empty(0, dtype=BACKBONE_DTYPE).element_size()


def normalize_map_device(value):
    if isinstance(value, int):
        return 'cuda:{}'.format(value)
    text = str(value).lower().replace('cuda ', 'cuda:')
    if text in {'0', '1'}:
        return 'cuda:' + text
    return text


def release_gpu_memory_before_load():
    gc.collect()
    for device_id in range(torch.cuda.device_count()):
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device_id)


def resolve_checkpoint_files(snapshot_dir):
    snapshot = Path(snapshot_dir)
    index_path = snapshot / 'model.safetensors.index.json'
    if index_path.is_file():
        weight_map = read_json(index_path).get('weight_map', {})
        names = sorted(set(weight_map.values()))
        if not names:
            raise RuntimeError('{} contains no weight_map entries.'.format(index_path))
    else:
        available = sorted(path.name for path in snapshot.glob('*.safetensors'))
        if not available:
            raise RuntimeError('No *.safetensors weight file was found in {}.'.format(snapshot))
        if 'model.safetensors' in available:
            names = ['model.safetensors']
        elif len(available) == 1:
            names = available
        else:
            raise RuntimeError('Ambiguous weight files in {}: {}.'.format(snapshot, available))
    files = []
    for name in names:
        candidate = snapshot / name
        # Containment is validated on the unresolved path on purpose: Hugging Face snapshot
        # directories symlink each entry into a sibling blobs/ store.
        if candidate.parent != snapshot or '/' in name or name.startswith('..'):
            raise RuntimeError('Checkpoint entry {!r} escapes the immutable snapshot directory.'.format(name))
        if not candidate.is_file():
            raise RuntimeError('Checkpoint file listed by the snapshot is missing: {}'.format(candidate))
        files.append(candidate)
    total_bytes = sum(path.stat().st_size for path in files)
    total_gib = total_bytes / 2 ** 30
    if not CHECKPOINT_MIN_TOTAL_GIB <= total_gib <= CHECKPOINT_MAX_TOTAL_GIB:
        raise RuntimeError('Implausible checkpoint size {:.2f} GiB for {}.'.format(total_gib, snapshot))
    return files, total_bytes


def tensor_static_bytes(tensor):
    element_bytes = DTYPE_ELEMENT_BYTES if tensor.is_floating_point() else tensor.element_size()
    return tensor.numel() * element_bytes


def module_static_bytes(module):
    total = 0
    for _, parameter in module.named_parameters(recurse=True):
        total += tensor_static_bytes(parameter)
    for _, submodule in module.named_modules():
        for buffer_name, buffer in submodule._buffers.items():
            if buffer is None or buffer_name in submodule._non_persistent_buffers_set:
                continue
            total += tensor_static_bytes(buffer)
    return total


def root_tensor_names_and_bytes(model):
    names = []
    total = 0
    for name, parameter in model._parameters.items():
        if parameter is None:
            continue
        names.append(name)
        total += tensor_static_bytes(parameter)
    for name, buffer in model._buffers.items():
        if buffer is None or name in model._non_persistent_buffers_set:
            continue
        names.append(name)
        total += tensor_static_bytes(buffer)
    return names, total


def plan_two_gpu_placement(model, block_attribute='blocks'):
    """Most balanced CONTIGUOUS block split, emitted as an explicit device-map dict."""
    blocks = getattr(model, block_attribute, None)
    if blocks is None or len(blocks) < 2:
        raise RuntimeError('The timm model does not expose a usable ordered "{}" list.'.format(block_attribute))
    num_blocks = len(blocks)

    root_names, root_bytes = root_tensor_names_and_bytes(model)
    block_bytes = [module_static_bytes(block) for block in blocks]
    input_side = []
    output_side = []
    for child_name, child in model.named_children():
        if child_name == block_attribute:
            continue
        if child_name in OUTPUT_SIDE_MODULE_NAMES:
            output_side.append((child_name, module_static_bytes(child)))
        else:
            input_side.append((child_name, module_static_bytes(child)))
    input_bytes = root_bytes + sum(size for _, size in input_side)
    output_bytes = sum(size for _, size in output_side)

    candidates = []
    for split in range(1, num_blocks):
        gpu0 = input_bytes + sum(block_bytes[:split])
        gpu1 = output_bytes + sum(block_bytes[split:])
        candidates.append((max(gpu0, gpu1), gpu0, split, gpu1))
    candidates.sort()
    _, gpu0_bytes, split_index, gpu1_bytes = candidates[0]

    device_map = {}
    for name in root_names:
        device_map[name] = 'cuda:0'
    for child_name, _ in input_side:
        device_map[child_name] = 'cuda:0'
    for index in range(num_blocks):
        device_map['{}.{}'.format(block_attribute, index)] = 'cuda:0' if index < split_index else 'cuda:1'
    for child_name, _ in output_side:
        device_map[child_name] = 'cuda:1'

    if '' in device_map:
        raise RuntimeError('The device map must not contain a root "" key; children would inherit GPU 0.')
    if set(device_map.values()) != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('The planned device map does not span both GPUs.')

    return {
        'strategy': SHARDING_STRATEGY,
        'block_attribute': block_attribute,
        'transformer_block_count': num_blocks,
        'contiguous_split_index': split_index,
        'cuda_0_block_range': [0, split_index - 1],
        'cuda_1_block_range': [split_index, num_blocks - 1],
        'root_tensor_names': root_names,
        'input_side_modules': [name for name, _ in input_side],
        'output_side_modules': [name for name, _ in output_side],
        'estimated_static_bytes': {'cuda:0': int(gpu0_bytes), 'cuda:1': int(gpu1_bytes)},
        'estimated_static_gib': {'cuda:0': gpu0_bytes / 2 ** 30, 'cuda:1': gpu1_bytes / 2 ** 30},
        'target_dtype': BACKBONE_DTYPE_NAME,
        'static_memory_limit_gib': GPU_STATIC_MEMORY_LIMIT_GIB,
        'min_runtime_headroom_gib': GPU_MIN_RUNTIME_HEADROOM_GIB,
        'device_map': device_map,
    }


def enforce_placement_budget(plan):
    limit_bytes = GPU_STATIC_MEMORY_LIMIT_GIB * 2 ** 30
    budget_rows = []
    for device_id in (0, 1):
        device_key = 'cuda:{}'.format(device_id)
        planned_bytes = plan['estimated_static_bytes'][device_key]
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        headroom_gib = (total_bytes - planned_bytes) / 2 ** 30
        budget_rows.append({
            'device': device_key,
            'planned_static_gib': planned_bytes / 2 ** 30,
            'device_total_gib': total_bytes / 2 ** 30,
            'device_free_gib': free_bytes / 2 ** 30,
            'projected_headroom_gib': headroom_gib,
        })
        if planned_bytes > limit_bytes:
            raise RuntimeError(
                'Planned static memory on {} is {:.2f} GiB, above the {:.2f} GiB ceiling.'.format(
                    device_key, planned_bytes / 2 ** 30, GPU_STATIC_MEMORY_LIMIT_GIB
                ) + '\n' + DISPATCH_FAILURE_HINT
            )
        if headroom_gib < GPU_MIN_RUNTIME_HEADROOM_GIB:
            raise RuntimeError(
                'Planned static memory on {} would leave only {:.2f} GiB of runtime headroom.'.format(
                    device_key, headroom_gib
                ) + '\n' + DISPATCH_FAILURE_HINT
            )
        if free_bytes < planned_bytes:
            raise RuntimeError(
                '{} currently has only {:.2f} GiB free but the plan needs {:.2f} GiB. Restart the session.'.format(
                    device_key, free_bytes / 2 ** 30, planned_bytes / 2 ** 30
                ) + '\n' + DISPATCH_FAILURE_HINT
            )
    plan['budget_rows'] = budget_rows
    return plan


def device_for_tensor_name(tensor_name, device_map):
    """Longest-prefix module match, mirroring Accelerate's own resolution order."""
    best_name = None
    best_device = None
    for module_name, device in device_map.items():
        if tensor_name == module_name or tensor_name.startswith(module_name + '.'):
            if best_name is None or len(module_name) > len(best_name):
                best_name = module_name
                best_device = device
    if best_device is None:
        raise RuntimeError('No device assignment covers checkpoint tensor {!r}.'.format(tensor_name))
    return best_device


def stream_checkpoint_into_model(model, checkpoint_files, device_map, dtype):
    """Read one tensor on the host, cast to FP16 on the host, send only to its target GPU.

    load_checkpoint_and_dispatch is deliberately not used: Accelerate stages the whole
    per-device slice of the single FP32 safetensors file on that GPU before the down-cast,
    which overflows a 14.56 GiB T4.
    """
    model_tensor_names = set(model.state_dict().keys())
    checkpoint_names = set()
    loaded = 0
    for checkpoint_file in checkpoint_files:
        with safe_open(str(checkpoint_file), framework='pt', device='cpu') as handle:
            keys = sorted(handle.keys())
            checkpoint_names.update(keys)
            for key in tqdm(keys, desc='streaming {}'.format(checkpoint_file.name)):
                if key not in model_tensor_names:
                    continue
                target_device = device_for_tensor_name(key, device_map)
                tensor = handle.get_tensor(key)
                if tensor.is_floating_point():
                    tensor = tensor.to(dtype)
                set_module_tensor_to_device(model, key, target_device, value=tensor, dtype=dtype)
                del tensor
                loaded += 1
    return {
        'tensors_loaded': loaded,
        'missing_keys': sorted(model_tensor_names - checkpoint_names),
        'unexpected_keys': sorted(checkpoint_names - model_tensor_names),
    }


def build_timm_backbone(snapshot_dir):
    with init_empty_weights(include_buffers=False):
        model = timm.create_model(TIMM_MODEL_NAME, pretrained=False, num_classes=0, img_size=IMAGE_SIZE)
    blocks = getattr(model, 'blocks', None)
    if blocks is None or len(blocks) < 2:
        raise RuntimeError('Could not resolve an ordered transformer block list on the timm model.')
    block_class_name = type(blocks[0]).__name__

    checkpoint_files, checkpoint_bytes = resolve_checkpoint_files(snapshot_dir)
    plan = plan_two_gpu_placement(model)
    plan['checkpoint_files'] = [str(path) for path in checkpoint_files]
    plan['checkpoint_total_gib'] = checkpoint_bytes / 2 ** 30
    plan['no_split_module_classes'] = [block_class_name]

    release_gpu_memory_before_load()
    plan['memory_before_load'] = cuda_memory_snapshot('before_checkpoint_load')
    write_json_atomic(plan, ARTIFACT_DIR / 'planned_device_map.json')
    plan = enforce_placement_budget(plan)
    write_json_atomic(plan, ARTIFACT_DIR / 'planned_device_map.json')

    print('Planned contiguous split: blocks {}-{} -> cuda:0, blocks {}-{} -> cuda:1'.format(
        plan['cuda_0_block_range'][0], plan['cuda_0_block_range'][1],
        plan['cuda_1_block_range'][0], plan['cuda_1_block_range'][1],
    ))
    print('Estimated static FP16: cuda:0 {:.2f} GiB, cuda:1 {:.2f} GiB (checkpoint on disk {:.2f} GiB)'.format(
        plan['estimated_static_gib']['cuda:0'], plan['estimated_static_gib']['cuda:1'], plan['checkpoint_total_gib'],
    ))

    try:
        load_report = stream_checkpoint_into_model(model, checkpoint_files, plan['device_map'], BACKBONE_DTYPE)
        if load_report['missing_keys']:
            raise RuntimeError(
                'The checkpoint does not provide {} model tensors, for example {}.'.format(
                    len(load_report['missing_keys']), load_report['missing_keys'][:10]
                )
            )
        if load_report['unexpected_keys']:
            print('Note: {} checkpoint keys are unused by this model, for example {}.'.format(
                len(load_report['unexpected_keys']), load_report['unexpected_keys'][:5]
            ))
        remaining_meta = [
            name
            for name, tensor in list(model.named_parameters()) + list(model.named_buffers())
            if tensor.device.type == 'meta'
        ]
        if remaining_meta:
            raise RuntimeError('These tensors are still on meta: {}.'.format(remaining_meta[:20]))
        model = dispatch_model(model, device_map=plan['device_map'], force_hooks=True)
    except Exception as exc:
        write_json_atomic(
            {
                'stage': 'timm_checkpoint_streaming_and_dispatch',
                'error_type': type(exc).__name__,
                'error': str(exc),
                'planned_device_map': plan['device_map'],
                'estimated_static_gib': plan['estimated_static_gib'],
                'memory_before_load': plan['memory_before_load'],
                'memory_after_failure': cuda_memory_snapshot('after_failure'),
                'action': DISPATCH_FAILURE_HINT,
            },
            ARTIFACT_DIR / 'model_load_failure_report.json',
        )
        raise RuntimeError(DISPATCH_FAILURE_HINT) from exc

    model.eval().requires_grad_(False)
    plan['load_report'] = load_report
    plan['memory_after_load'] = cuda_memory_snapshot('after_checkpoint_load')
    write_json_atomic(plan, ARTIFACT_DIR / 'planned_device_map.json')
    return model, list(model.blocks), block_class_name, plan


def audit_dispatched_backbone(model, blocks):
    raw_device_map = getattr(model, 'hf_device_map', None)
    if not isinstance(raw_device_map, dict) or not raw_device_map:
        raise RuntimeError('Accelerate did not attach an hf_device_map.' + '\n' + DISPATCH_FAILURE_HINT)
    device_map = {name: normalize_map_device(value) for name, value in raw_device_map.items()}
    mapped_devices = set(device_map.values())
    forbidden = sorted(value for value in mapped_devices if 'cpu' in value or 'disk' in value or 'meta' in value)
    if forbidden:
        raise RuntimeError('CPU/disk/meta placement detected: {}.'.format(forbidden) + '\n' + DISPATCH_FAILURE_HINT)
    if mapped_devices != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('Device map must span exactly cuda:0 and cuda:1; got {}.'.format(sorted(mapped_devices)))

    named_tensors = list(model.named_parameters()) + list(model.named_buffers())
    meta_tensors = [name for name, tensor in named_tensors if tensor.device.type == 'meta']
    if meta_tensors:
        raise RuntimeError('These tensors are still on the meta device: {}.'.format(meta_tensors[:20]))
    parameter_devices = {str(parameter.device) for parameter in model.parameters()}
    if parameter_devices != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('Parameters are not split across exactly both GPUs: {}.'.format(sorted(parameter_devices)))
    parameter_dtypes = {parameter.dtype for parameter in model.parameters() if parameter.is_floating_point()}
    if parameter_dtypes != {BACKBONE_DTYPE}:
        raise RuntimeError('Floating parameters are not uniformly FP16: {}.'.format(sorted(map(str, parameter_dtypes))))

    block_devices = []
    for block_index, block in enumerate(blocks):
        devices = {str(parameter.device) for parameter in block.parameters()}
        if len(devices) != 1:
            raise RuntimeError('Transformer block {} is split across devices {}.'.format(block_index, sorted(devices)))
        block_devices.append(devices.pop())
    transitions = [index for index in range(1, len(block_devices)) if block_devices[index] != block_devices[index - 1]]
    if len(transitions) != 1:
        raise RuntimeError(
            'Transformer blocks must form exactly two contiguous device ranges; found {} transitions.'.format(
                len(transitions)
            )
        )
    if block_devices[0] != 'cuda:0' or block_devices[-1] != 'cuda:1':
        raise RuntimeError('Contiguous block ranges must run cuda:0 then cuda:1.')

    hooked_modules = [name for name, module in model.named_modules() if hasattr(module, '_hf_hook')]
    if not hooked_modules:
        raise RuntimeError('No Accelerate hooks are attached.' + '\n' + DISPATCH_FAILURE_HINT)

    for device_id in (0, 1):
        free_bytes, _ = torch.cuda.mem_get_info(device_id)
        if free_bytes / 2 ** 30 < GPU_MIN_RUNTIME_HEADROOM_GIB:
            raise RuntimeError(
                'cuda:{} has only {:.2f} GiB free after loading.'.format(device_id, free_bytes / 2 ** 30)
            )

    forbidden_wrapper_names = {
        ''.join(('Data', 'Parallel')),
        ''.join(('Distributed', 'Data', 'Parallel')),
        ''.join(('Fully', 'Sharded', 'Data', 'Parallel')),
    }
    if type(model).__name__ in forbidden_wrapper_names:
        raise RuntimeError('Replication or distributed wrapper detected; only model sharding is allowed.')
    return device_map, block_devices, len(hooked_modules)


MODEL_LOAD_STARTED_AT = time.perf_counter()
BACKBONE, BACKBONE_BLOCKS, BACKBONE_BLOCK_CLASS, PLACEMENT_PLAN = build_timm_backbone(MODEL_SNAPSHOT_DIR)
MODEL_LOAD_SECONDS = time.perf_counter() - MODEL_LOAD_STARTED_AT
del HF_TOKEN_VALUE

DEVICE_MAP, BLOCK_DEVICES, HOOKED_MODULE_COUNT = audit_dispatched_backbone(BACKBONE, BACKBONE_BLOCKS)
BACKBONE_PARAMETER_COUNT = sum(parameter.numel() for parameter in BACKBONE.parameters())
BACKBONE_TRAINABLE_PARAMETER_COUNT = sum(
    parameter.numel() for parameter in BACKBONE.parameters() if parameter.requires_grad
)
if BACKBONE_TRAINABLE_PARAMETER_COUNT != 0:
    raise RuntimeError('The pretrained backbone is not completely frozen.')
if BACKBONE.training:
    raise RuntimeError('The backbone must stay in evaluation mode.')

# Token geometry read off real model attributes, never assumed.
NUM_PREFIX_TOKENS = getattr(BACKBONE, 'num_prefix_tokens', None)
if NUM_PREFIX_TOKENS is None:
    raise RuntimeError('The timm model does not expose num_prefix_tokens.')
NUM_PREFIX_TOKENS = int(NUM_PREFIX_TOKENS)
register_parameter = getattr(BACKBONE, 'reg_token', None)
NUM_REGISTER_TOKENS = int(register_parameter.shape[1]) if register_parameter is not None else NUM_PREFIX_TOKENS - 1
HIDDEN_SIZE = int(BACKBONE.num_features)
FINAL_NORM_MODULE = BACKBONE.norm
PATCH_INPUT_MODULE = BACKBONE.patch_embed.proj
MODEL_INPUT_DEVICE = next(PATCH_INPUT_MODULE.parameters()).device
FINAL_NORM_DEVICE = next(FINAL_NORM_MODULE.parameters()).device
SEQUENCE_TOKEN_COUNT = NUM_PREFIX_TOKENS + EXPECTED_PATCH_TOKENS

if HIDDEN_SIZE != EXPECTED_HIDDEN_SIZE:
    raise RuntimeError('Runtime hidden size {} != expected {}.'.format(HIDDEN_SIZE, EXPECTED_HIDDEN_SIZE))
if NUM_PREFIX_TOKENS != EXPECTED_PREFIX_TOKENS or NUM_REGISTER_TOKENS != EXPECTED_REGISTER_TOKENS:
    raise RuntimeError(
        'Runtime prefix/register token counts ({}, {}) != expected ({}, {}).'.format(
            NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS, EXPECTED_PREFIX_TOKENS, EXPECTED_REGISTER_TOKENS
        )
    )
if SEQUENCE_TOKEN_COUNT != EXPECTED_SEQUENCE_TOKENS:
    raise RuntimeError('Runtime sequence token count {} != expected.'.format(SEQUENCE_TOKEN_COUNT))
if str(FINAL_NORM_DEVICE) != BLOCK_DEVICES[-1]:
    raise RuntimeError('The final norm is not on the same GPU as the last transformer block.')

# ---------------------------------------------------------------------------
# WHAT "MODEL-NATIVE" ACTUALLY MEANS, read from timm rather than assumed.
# ---------------------------------------------------------------------------
TIMM_DATA_CONFIG = timm.data.resolve_model_data_config(BACKBONE)
RESOLVED_MEAN = tuple(float(value) for value in TIMM_DATA_CONFIG['mean'])
RESOLVED_STD = tuple(float(value) for value in TIMM_DATA_CONFIG['std'])
RESOLVED_INPUT_SIZE = tuple(int(value) for value in TIMM_DATA_CONFIG['input_size'])
RESOLVED_INTERPOLATION = TIMM_DATA_CONFIG.get('interpolation')
RESOLVED_CROP_PCT = TIMM_DATA_CONFIG.get('crop_pct')

if not np.allclose(RESOLVED_MEAN, IMAGENET_MEAN, atol=1e-6) or not np.allclose(RESOLVED_STD, IMAGENET_STD, atol=1e-6):
    raise RuntimeError('timm normalization {} / {} does not match the locked ImageNet constants.'.format(
        RESOLVED_MEAN, RESOLVED_STD
    ))
if RESOLVED_INPUT_SIZE != (3, IMAGE_SIZE, IMAGE_SIZE):
    raise RuntimeError('timm input size {} is not the native {}px contract.'.format(RESOLVED_INPUT_SIZE, IMAGE_SIZE))
if RESOLVED_INTERPOLATION != VARIANT_INTERPOLATION[NATIVE_VARIANT]:
    raise RuntimeError(
        'The whole point of this experiment is that "{}" is the model-native kernel, but timm resolved "{}". '
        'Stop and re-derive the hypothesis before spending GPU hours.'.format(
            VARIANT_INTERPOLATION[NATIVE_VARIANT], RESOLVED_INTERPOLATION
        )
    )

MODEL_MANIFEST = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'backend': 'timm',
    'repository': TIMM_HF_REPO_ID,
    'immutable_revision': RESOLVED_REVISION,
    'timm_model_name': TIMM_MODEL_NAME,
    'model_class': type(BACKBONE).__name__,
    'block_class': BACKBONE_BLOCK_CLASS,
    'parameter_count': BACKBONE_PARAMETER_COUNT,
    'trainable_parameter_count': BACKBONE_TRAINABLE_PARAMETER_COUNT,
    'transformer_block_count': len(BACKBONE_BLOCKS),
    'hidden_size': HIDDEN_SIZE,
    'num_prefix_tokens': NUM_PREFIX_TOKENS,
    'num_register_tokens': NUM_REGISTER_TOKENS,
    'sequence_token_count': SEQUENCE_TOKEN_COUNT,
    'token_layout': 'CLS, register tokens, true patch tokens',
    'backbone_dtype': BACKBONE_DTYPE_NAME,
    'model_input_device': str(MODEL_INPUT_DEVICE),
    'final_norm_device': str(FINAL_NORM_DEVICE),
    'placement_strategy': SHARDING_STRATEGY,
    'block_devices': BLOCK_DEVICES,
    'accelerate_hooked_module_count': HOOKED_MODULE_COUNT,
    'automatic_device_map_string_used': False,
    'cpu_or_disk_offload': False,
    'data_parallel_used': False,
    'model_download_seconds': MODEL_DOWNLOAD_SECONDS,
    'model_load_seconds': MODEL_LOAD_SECONDS,
    'timm_resolved_data_config': {
        'mean': list(RESOLVED_MEAN),
        'std': list(RESOLVED_STD),
        'input_size': list(RESOLVED_INPUT_SIZE),
        'interpolation': RESOLVED_INTERPOLATION,
        'crop_pct': RESOLVED_CROP_PCT,
    },
    'crop_pct_note': (
        'timm default eval transform would resize the short side to input_size/crop_pct and then center crop. '
        'V13 keeps the V7/V10 direct-square geometry on purpose so the only change is the kernel; the timm '
        'crop geometry is a separate, untested ablation and is NOT applied here.'
    ),
    'environment': ENVIRONMENT_INFO,
    'gpus': GPU_RUNTIME_INFO,
}
write_json_atomic(MODEL_MANIFEST, ARTIFACT_DIR / 'model_manifest.json')
write_json_atomic(
    {
        'device_map': DEVICE_MAP,
        'devices': sorted(set(DEVICE_MAP.values())),
        'block_devices': BLOCK_DEVICES,
        'actual_cuda_memory': cuda_memory_snapshot('after_audit'),
        'accelerate_hooked_module_count': HOOKED_MODULE_COUNT,
    },
    ARTIFACT_DIR / 'device_map.json',
)

print()
print('Backbone           :', type(BACKBONE).__name__)
print('Parameters         : {:,} (trainable {:,})'.format(BACKBONE_PARAMETER_COUNT, BACKBONE_TRAINABLE_PARAMETER_COUNT))
print('Block placement    : cuda:0 x{} then cuda:1 x{}'.format(
    BLOCK_DEVICES.count('cuda:0'), BLOCK_DEVICES.count('cuda:1')
))
print('Token contract     : {} = {} prefix ({} register) + {} patch, hidden {}'.format(
    SEQUENCE_TOKEN_COUNT, NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS, EXPECTED_PATCH_TOKENS, HIDDEN_SIZE
))
print('Accelerate hooks   :', HOOKED_MODULE_COUNT, 'modules')
print('timm native config : interpolation={} crop_pct={} input_size={}'.format(
    RESOLVED_INTERPOLATION, RESOLVED_CROP_PCT, RESOLVED_INPUT_SIZE
))
print('  -> confirms "{}" is the model-native kernel.'.format(RESOLVED_INTERPOLATION))
for row in cuda_memory_snapshot('after_audit'):
    print('  cuda:{device_id} allocated {allocated_gib:.2f} GiB, free {free_gib:.2f} GiB'.format(**row))

In [ ]:
def finalize_feature_storage(pooled_fp32, expected_rows=None):
    """Single canonical FP32 -> CPU FP16 storage path, shared by extraction and preflight."""
    if pooled_fp32.dtype != POOLING_COMPUTE_DTYPE:
        raise RuntimeError('Feature storage expects FP32 pooled features, got {}.'.format(pooled_fp32.dtype))
    if pooled_fp32.ndim != 3 or tuple(pooled_fp32.shape[1:]) != (NUM_BRANCHES, HIDDEN_SIZE):
        raise RuntimeError('Pooled feature shape {} is invalid.'.format(tuple(pooled_fp32.shape)))
    if expected_rows is not None and pooled_fp32.shape[0] != expected_rows:
        raise RuntimeError('Pooled feature row count {} != expected {}.'.format(pooled_fp32.shape[0], expected_rows))
    if not torch.isfinite(pooled_fp32).all().item():
        raise RuntimeError('Pooled features contain NaN or Inf; extraction is aborted.')
    pooled_max_abs = float(pooled_fp32.abs().max().item())
    if not math.isfinite(pooled_max_abs) or pooled_max_abs > FEATURE_MAX_ABS_LIMIT:
        raise RuntimeError('Pooled feature range is implausible: {}.'.format(pooled_max_abs))
    return pooled_fp32.detach().to(device='cpu', dtype=FEATURE_STORAGE_DTYPE)


def normalized_tokens(model_input):
    """timm forward_features ends with self.norm, so the official final norm is applied exactly once."""
    tokens = BACKBONE.forward_features(model_input)
    if not torch.is_tensor(tokens):
        raise RuntimeError('forward_features did not return a token tensor.')
    if tokens.ndim != 3 or tokens.shape[1] != SEQUENCE_TOKEN_COUNT or tokens.shape[2] != HIDDEN_SIZE:
        raise RuntimeError(
            'forward_features returned shape {}, expected [B, {}, {}].'.format(
                tuple(tokens.shape), SEQUENCE_TOKEN_COUNT, HIDDEN_SIZE
            )
        )
    if tokens.dtype != BACKBONE_DTYPE:
        raise RuntimeError('Normalized tokens are not FP16 ({}).'.format(tokens.dtype))
    return tokens


def pool_two_branches(tokens):
    """[B, 261, 4096] normalized tokens -> [B, 2, 4096] FP32.

    Branch 0: prefix token 0 (CLS).
    Branch 1: mean over the 256 true patch tokens. The 4 register tokens sit at prefix
    positions 1..4 and are sliced away, so they never enter any patch statistic.
    """
    cls_fp32 = tokens[:, 0, :].to(POOLING_COMPUTE_DTYPE)
    patches_fp32 = tokens[:, NUM_PREFIX_TOKENS:, :].to(POOLING_COMPUTE_DTYPE)
    if patches_fp32.shape[1] != EXPECTED_PATCH_TOKENS:
        raise RuntimeError('True-patch token count is {}, expected {}.'.format(
            patches_fp32.shape[1], EXPECTED_PATCH_TOKENS
        ))
    return torch.stack((cls_fp32, patches_fp32.mean(dim=1)), dim=1)


def extract_features(pixel_values):
    """pixel_values: [B, 3, 256, 256] FP32 on CPU -> [B, 2, 4096] FP16 on CPU."""
    if pixel_values.ndim != 4 or tuple(pixel_values.shape[1:]) != (3, IMAGE_SIZE, IMAGE_SIZE):
        raise RuntimeError('Feature extraction expects [B, 3, {}, {}] input, got {}.'.format(
            IMAGE_SIZE, IMAGE_SIZE, tuple(pixel_values.shape)
        ))
    if pixel_values.dtype != torch.float32:
        raise RuntimeError('Feature extraction expects FP32 pixel values, got {}.'.format(pixel_values.dtype))
    with torch.inference_mode():
        model_input = pixel_values.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE, non_blocking=True)
        tokens = normalized_tokens(model_input)
        pooled_fp32 = pool_two_branches(tokens)
        return finalize_feature_storage(pooled_fp32, expected_rows=pixel_values.shape[0])


def deterministic_pattern(width, height, phase):
    x = np.arange(width, dtype=np.uint32)[None, :]
    y = np.arange(height, dtype=np.uint32)[:, None]
    red = np.broadcast_to((x * 7 + phase * 31) % 256, (height, width))
    green = np.broadcast_to((y * 11 + phase * 17) % 256, (height, width))
    checker = (((x // 9 + y // 13 + phase) % 2) * 127).astype(np.uint32)
    blue = (x * 3 + y * 5 + checker + phase * 23) % 256
    return Image.fromarray(np.stack((red, green, blue), axis=2).astype(np.uint8))


def deterministic_sample_paths(frame, count):
    ranked = sorted(
        (hashlib.sha256(image_id.encode('utf-8')).hexdigest(), path)
        for image_id, path in zip(frame['image_id'], frame['path'])
    )
    if count > len(ranked):
        raise RuntimeError('Sample is larger than the indexed train set.')
    return [path for _, path in ranked[:count]]


def run_preflight():
    stage = 'A_token_contract'
    report = {
        'schema_version': 1,
        'revision_marker': REVISION_MARKER,
        'pass': False,
        'failed_stage': None,
        'model_revision': RESOLVED_REVISION,
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
        'feature_batch_size': FEATURE_BATCH_SIZE,
        'stages': {},
    }
    try:
        synthetic_specs = ((317, 193, 1), (211, 359, 2))
        synthetic_images = [deterministic_pattern(*spec) for spec in synthetic_specs]
        control_transform = VARIANT_TRANSFORMS[CONTROL_VARIANT]
        synthetic_batch = torch.stack([control_transform(image) for image in synthetic_images])
        with torch.inference_mode():
            model_input = synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE)
            tokens = normalized_tokens(model_input)
            tokens_finite = bool(torch.isfinite(tokens.float()).all().item())
            # Applying the final norm a second time must visibly change the tensor. This proves the
            # tokens are already post-norm output, so V13 never double-normalises.
            double_normed = FINAL_NORM_MODULE(tokens.to(FINAL_NORM_DEVICE))
            double_norm_delta = float((double_normed.float() - tokens.to(FINAL_NORM_DEVICE).float()).abs().max().item())
        report['stages'][stage] = {
            'pass': bool(tokens_finite and double_norm_delta > 0.0),
            'sequence_token_count': int(tokens.shape[1]),
            'prefix_token_count': NUM_PREFIX_TOKENS,
            'register_token_count': NUM_REGISTER_TOKENS,
            'true_patch_token_count': int(tokens.shape[1] - NUM_PREFIX_TOKENS),
            'hidden_size': int(tokens.shape[2]),
            'dtype': str(tokens.dtype),
            'tokens_finite': tokens_finite,
            'second_norm_application_max_abs_delta': double_norm_delta,
            'final_norm_applied_times': 1,
            'note': 'a non-zero delta proves forward_features already applied the norm; V13 never applies it again',
        }
        if not report['stages'][stage]['pass']:
            raise RuntimeError('Token contract or final-norm-once check failed.')

        stage = 'B_register_token_exclusion'
        with torch.inference_mode():
            model_input = synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE)
            tokens = normalized_tokens(model_input)
            pooled = pool_two_branches(tokens)
            true_patch_mean = tokens[:, NUM_PREFIX_TOKENS:, :].to(POOLING_COMPUTE_DTYPE).mean(dim=1)
            all_non_cls_mean = tokens[:, 1:, :].to(POOLING_COMPUTE_DTYPE).mean(dim=1)
            direct_cls = tokens[:, 0, :].to(POOLING_COMPUTE_DTYPE)
            cls_delta = float((pooled[:, 0] - direct_cls).abs().max().item())
            mean_delta = float((pooled[:, 1] - true_patch_mean).abs().max().item())
            register_delta = float((true_patch_mean - all_non_cls_mean).abs().max().item())
            storage = finalize_feature_storage(pooled, expected_rows=len(synthetic_images))
        branch_ok = cls_delta == 0.0 and mean_delta == 0.0
        register_ok = register_delta > 0.0
        storage_ok = (
            tuple(storage.shape) == (len(synthetic_images), NUM_BRANCHES, HIDDEN_SIZE)
            and storage.dtype == FEATURE_STORAGE_DTYPE
            and storage.device.type == 'cpu'
        )
        report['stages'][stage] = {
            'pass': bool(branch_ok and register_ok and storage_ok),
            'branch0_vs_direct_cls_max_abs_delta': cls_delta,
            'branch1_vs_direct_true_patch_mean_max_abs_delta': mean_delta,
            'register_exclusion_max_abs_delta': register_delta,
            'register_exclusion_note': 'non-zero delta proves the 4 register tokens are excluded from the patch mean',
            'storage_shape': list(storage.shape),
            'storage_dtype': str(storage.dtype),
        }
        if not report['stages'][stage]['pass']:
            raise RuntimeError('Branch semantics, register exclusion or storage cast failed.')

        stage = 'C_interpolation_is_live'
        # The single experimental variable must actually change the pixels that reach the backbone.
        # A silent no-op here would burn two hours of GPU and produce a meaningless null result.
        probe_paths = deterministic_sample_paths(train_df, 64)
        pixel_deltas = []
        identical_count = 0
        for path in probe_paths:
            image = load_rgb_image(path)
            control_pixels = VARIANT_TRANSFORMS[CONTROL_VARIANT](image)
            native_pixels = VARIANT_TRANSFORMS[NATIVE_VARIANT](image)
            delta = float((control_pixels - native_pixels).abs().max().item())
            pixel_deltas.append(delta)
            if delta == 0.0:
                identical_count += 1
        with torch.inference_mode():
            control_batch = torch.stack([
                VARIANT_TRANSFORMS[CONTROL_VARIANT](load_rgb_image(path)) for path in probe_paths[:FEATURE_BATCH_SIZE]
            ])
            native_batch = torch.stack([
                VARIANT_TRANSFORMS[NATIVE_VARIANT](load_rgb_image(path)) for path in probe_paths[:FEATURE_BATCH_SIZE]
            ])
            control_features = extract_features(control_batch).to(POOLING_COMPUTE_DTYPE)
            native_features = extract_features(native_batch).to(POOLING_COMPUTE_DTYPE)
            feature_delta = float((control_features - native_features).abs().max().item())
            feature_cosine = float(
                torch.nn.functional.cosine_similarity(
                    control_features.flatten(1), native_features.flatten(1), dim=1
                ).mean().item()
            )
        interpolation_live = max(pixel_deltas) > 0.0 and identical_count < len(probe_paths) and feature_delta > 0.0
        report['stages'][stage] = {
            'pass': bool(interpolation_live),
            'probe_images': len(probe_paths),
            'images_with_identical_pixels': identical_count,
            'pixel_max_abs_delta_max': max(pixel_deltas),
            'pixel_max_abs_delta_mean': float(np.mean(pixel_deltas)),
            'feature_max_abs_delta': feature_delta,
            'feature_mean_cosine_similarity': feature_cosine,
            'note': 'proves bilinear and bicubic really produce different inputs and different features',
        }
        if not interpolation_live:
            raise RuntimeError(
                'The two variants produce identical pixels or identical features. The experimental variable '
                'is a no-op and the run would be meaningless. Check build_transform.'
            )

        stage = 'D_throughput'
        for device_id in (0, 1):
            torch.cuda.reset_peak_memory_stats(device_id)
        sample_paths = deterministic_sample_paths(train_df, BENCHMARK_WARMUP_IMAGES + BENCHMARK_TIMED_IMAGES)
        warmup_loader = DataLoader(
            ImageRows(sample_paths[:BENCHMARK_WARMUP_IMAGES], VARIANT_TRANSFORMS[CONTROL_VARIANT]),
            batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0,
        )
        for batch in warmup_loader:
            extract_features(batch)
        for device_id in (0, 1):
            torch.cuda.synchronize(device_id)
        timed_loader = DataLoader(
            ImageRows(sample_paths[BENCHMARK_WARMUP_IMAGES:], VARIANT_TRANSFORMS[CONTROL_VARIANT]),
            batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0,
        )
        timed_started = time.perf_counter()
        for batch in timed_loader:
            extract_features(batch)
        for device_id in (0, 1):
            torch.cuda.synchronize(device_id)
        timed_seconds = time.perf_counter() - timed_started
        images_per_second = BENCHMARK_TIMED_IMAGES / timed_seconds
        throughput_pass = images_per_second >= MIN_EXTRACTION_IMAGES_PER_SECOND
        report['stages'][stage] = {
            'pass': bool(throughput_pass),
            'timed_images': BENCHMARK_TIMED_IMAGES,
            'timed_seconds': timed_seconds,
            'images_per_second': images_per_second,
            'minimum_images_per_second': MIN_EXTRACTION_IMAGES_PER_SECOND,
            'estimated_hours_per_variant': EXPECTED_TRAIN_ROWS / images_per_second / 3600.0,
            'estimated_hours_both_variants': 2 * EXPECTED_TRAIN_ROWS / images_per_second / 3600.0,
            'estimated_test_minutes': EXPECTED_TEST_ROWS / images_per_second / 60.0,
            'gpu_memory': gpu_memory_rows(),
            'host_rss_gib': host_rss_gib(),
        }
        if not throughput_pass:
            raise RuntimeError('Measured {:.3f} images/s is below the locked gate.'.format(images_per_second))

        report['pass'] = True
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        return report
    except torch.cuda.OutOfMemoryError as exc:
        report['failed_stage'] = stage
        report['error_type'] = type(exc).__name__
        report['error'] = str(exc)
        report['action'] = 'Lower FEATURE_BATCH_SIZE (try 2, then 1) and rerun from the top.'
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        raise RuntimeError(
            'GPU OOM during preflight stage {} at FEATURE_BATCH_SIZE={}. Lower it to 2 or 1.'.format(
                stage, FEATURE_BATCH_SIZE
            )
        ) from exc
    except Exception as exc:
        report['failed_stage'] = stage
        report['error_type'] = type(exc).__name__
        report['error'] = str(exc)
        report['action'] = 'Resolve the reported stage and rerun. Full feature extraction has not started.'
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        raise RuntimeError('Preflight failed at stage {}; inspect {}.'.format(stage, PREFLIGHT_REPORT_PATH)) from exc


PREFLIGHT_REPORT = run_preflight()
PREFLIGHT_SIGNATURE = sha256_json({
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'model_revision': RESOLVED_REVISION,
    'branch_names': list(BRANCH_NAMES),
    'token_contract': [SEQUENCE_TOKEN_COUNT, NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS, EXPECTED_PATCH_TOKENS, HIDDEN_SIZE],
})

interp_stage = PREFLIGHT_REPORT['stages']['C_interpolation_is_live']
throughput_stage = PREFLIGHT_REPORT['stages']['D_throughput']
print()
print('Preflight PASSED. Signature:', PREFLIGHT_SIGNATURE)
print('Final norm applied exactly once (second application delta {:.4f})'.format(
    PREFLIGHT_REPORT['stages']['A_token_contract']['second_norm_application_max_abs_delta']
))
print('Register exclusion delta : {:.4f}'.format(
    PREFLIGHT_REPORT['stages']['B_register_token_exclusion']['register_exclusion_max_abs_delta']
))
print('Interpolation is live    : {} of {} probe images differ; max pixel delta {:.4f}, feature cosine {:.6f}'.format(
    interp_stage['probe_images'] - interp_stage['images_with_identical_pixels'],
    interp_stage['probe_images'], interp_stage['pixel_max_abs_delta_max'], interp_stage['feature_mean_cosine_similarity'],
))
print('Throughput               : {:.3f} img/s -> {:.2f} h per variant, {:.2f} h for both'.format(
    throughput_stage['images_per_second'],
    throughput_stage['estimated_hours_per_variant'],
    throughput_stage['estimated_hours_both_variants'],
))

In [ ]:
CACHE_SCHEMA_VERSION = 1


def build_cache_identity(variant_name, split, row_signature):
    identity = {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'identity_scope': 'frozen_backbone_features_only_no_head_hyperparameters',
        'backend': 'timm',
        'model_repository': TIMM_HF_REPO_ID,
        'model_revision': RESOLVED_REVISION,
        'timm_model_name': TIMM_MODEL_NAME,
        'timm_version': ENVIRONMENT_INFO['timm_version'],
        'torch_version': ENVIRONMENT_INFO['torch_version'],
        'cuda_version': ENVIRONMENT_INFO['cuda_version'],
        'accelerate_version': ENVIRONMENT_INFO['accelerate_version'],
        'gpu_names': [info['name'] for info in GPU_RUNTIME_INFO],
        'image_size': IMAGE_SIZE,
        'variant': variant_name,
        'interpolation': VARIANT_INTERPOLATION[variant_name],
        'preprocess_signature': VARIANT_PREPROCESS_SIGNATURE[variant_name],
        'backbone_dtype': BACKBONE_DTYPE_NAME,
        'pooling_compute_dtype': POOLING_COMPUTE_DTYPE_NAME,
        'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
        'readout': READOUT_DESCRIPTION,
        'branch_names': list(BRANCH_NAMES),
        'token_contract': {
            'sequence_token_count': SEQUENCE_TOKEN_COUNT,
            'prefix_token_count': NUM_PREFIX_TOKENS,
            'register_token_count': NUM_REGISTER_TOKENS,
            'patch_token_count': EXPECTED_PATCH_TOKENS,
            'hidden_size': HIDDEN_SIZE,
            'register_tokens_excluded': True,
        },
        'raw_patch_tokens_stored': False,
        'preflight_signature': PREFLIGHT_SIGNATURE,
        'split': split,
    }
    identity.update(row_signature)
    return identity


def shard_name(split, start, end):
    return '{}_{:06d}_{:06d}.pt'.format(split, start, end)


def expected_shard_metadata(variant_name, split, cache_signature, frame, start, end):
    window = frame.iloc[start:end]
    labels = window['label'].astype(int).tolist() if 'label' in window.columns else None
    folds = window['fold'].astype(int).tolist() if 'fold' in window.columns else None
    return {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'variant': variant_name,
        'cache_signature': cache_signature,
        'split': split,
        'start': int(start),
        'end': int(end),
        'rows': int(end - start),
        'image_ids_sha256': sha256_json(window['image_id'].tolist()),
        'labels_sha256': sha256_json(labels) if labels is not None else None,
        'folds_sha256': sha256_json(folds) if folds is not None else None,
        'feature_shape': [int(end - start), NUM_BRANCHES, HIDDEN_SIZE],
        'feature_dtype': FEATURE_STORAGE_DTYPE_NAME,
        'branch_names': list(BRANCH_NAMES),
    }


def validate_shard(path, expected_metadata, frame, start, end):
    """Every failure raises RuntimeError so the caller can safely recompute the shard."""
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
    except Exception as exc:
        raise RuntimeError('Feature shard cannot be loaded: {}'.format(path)) from exc
    if not isinstance(payload, dict) or payload.get('metadata') != expected_metadata:
        raise RuntimeError('Feature shard metadata is stale or mismatched: {}'.format(path))
    features = payload.get('features')
    if not torch.is_tensor(features) or tuple(features.shape) != tuple(expected_metadata['feature_shape']):
        raise RuntimeError('Feature shard tensor shape is invalid: {}'.format(path))
    if features.dtype != FEATURE_STORAGE_DTYPE:
        raise RuntimeError('Feature shard dtype is invalid: {}'.format(path))
    if not torch.isfinite(features.to(POOLING_COMPUTE_DTYPE)).all().item():
        raise RuntimeError('Feature shard contains NaN or Inf: {}'.format(path))
    window = frame.iloc[start:end]
    if payload.get('image_ids') != window['image_id'].tolist():
        raise RuntimeError('Feature shard image IDs are misaligned: {}'.format(path))
    return payload


def extract_shard(variant_name, frame, start, end):
    window = frame.iloc[start:end]
    features = torch.empty((end - start, NUM_BRANCHES, HIDDEN_SIZE), dtype=FEATURE_STORAGE_DTYPE)
    loader = DataLoader(
        ImageRows(window['path'].tolist(), VARIANT_TRANSFORMS[variant_name]),
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=FEATURE_LOADER_WORKERS,
        pin_memory=True,
        persistent_workers=False,
    )
    cursor = 0
    for batch in loader:
        pooled = extract_features(batch)
        features[cursor:cursor + pooled.shape[0]] = pooled
        cursor += pooled.shape[0]
        del pooled
    if cursor != end - start:
        raise RuntimeError('Shard extraction produced {} rows, expected {}.'.format(cursor, end - start))
    del loader
    return features


def build_feature_cache(variant_name, split, frame):
    """Resumable, atomic, signature-bound feature cache for one variant and one split."""
    row_signature = {
        'rows': int(len(frame)),
        'image_ids_sha256': sha256_json(frame['image_id'].tolist()),
    }
    if 'label' in frame.columns:
        row_signature['labels_sha256'] = sha256_json(frame['label'].astype(int).tolist())
    if 'fold' in frame.columns:
        row_signature['folds_sha256'] = sha256_json(frame['fold'].astype(int).tolist())
    identity = build_cache_identity(variant_name, split, row_signature)
    cache_signature = sha256_json(identity)

    shard_dir = CACHE_DIR / variant_name / split
    shard_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = ARTIFACT_DIR / 'cache_manifest_{}_{}.json'.format(variant_name, split)

    features = torch.empty((len(frame), NUM_BRANCHES, HIDDEN_SIZE), dtype=FEATURE_STORAGE_DTYPE)
    boundaries = [
        (start, min(start + FEATURE_SHARD_SIZE, len(frame)))
        for start in range(0, len(frame), FEATURE_SHARD_SIZE)
    ]
    shard_records = []
    reused_rows = 0
    written_rows = 0
    extraction_seconds = 0.0
    started_at = time.perf_counter()

    progress = tqdm(boundaries, desc='{} / {} cache'.format(variant_name, split))
    for start, end in progress:
        metadata = expected_shard_metadata(variant_name, split, cache_signature, frame, start, end)
        shard_path = shard_dir / shard_name(split, start, end)
        payload = None
        reused = False
        if shard_path.is_file():
            try:
                payload = validate_shard(shard_path, metadata, frame, start, end)
                reused = True
            except RuntimeError:
                payload = None
        if payload is None:
            shard_started = time.perf_counter()
            shard_features = extract_shard(variant_name, frame, start, end)
            extraction_seconds += time.perf_counter() - shard_started
            payload = {
                'metadata': metadata,
                'features': shard_features,
                'image_ids': frame.iloc[start:end]['image_id'].tolist(),
            }
            if 'label' in frame.columns:
                payload['labels'] = torch.tensor(frame.iloc[start:end]['label'].to_numpy(), dtype=torch.long)
            if 'fold' in frame.columns:
                payload['folds'] = torch.tensor(frame.iloc[start:end]['fold'].to_numpy(), dtype=torch.long)
            atomic_torch_save(payload, shard_path)
            written_rows += end - start
        else:
            reused_rows += end - start
        features[start:end] = payload['features']
        shard_records.append({'file': shard_path.name, 'start': start, 'end': end, 'reused': reused})
        del payload
        progress.set_postfix(reused=reused_rows, written=written_rows)

    if not torch.isfinite(features.to(POOLING_COMPUTE_DTYPE)).all().item():
        raise RuntimeError('The assembled feature cache contains NaN or Inf.')

    manifest = {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'variant': variant_name,
        'split': split,
        'complete': True,
        'rows': int(len(frame)),
        'cache_signature': cache_signature,
        'cache_identity': identity,
        'shard_size': FEATURE_SHARD_SIZE,
        'shard_count': len(boundaries),
        'rows_written_this_session': written_rows,
        'rows_reused_this_session': reused_rows,
        'feature_shape': [int(len(frame)), NUM_BRANCHES, HIDDEN_SIZE],
        'feature_bytes': int(features.numel() * features.element_size()),
        'new_extraction_seconds': extraction_seconds,
        'new_extraction_images_per_second': (written_rows / extraction_seconds) if extraction_seconds > 0 else None,
        'elapsed_seconds_this_session': time.perf_counter() - started_at,
        'gpu_memory': gpu_memory_rows(),
        'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
        'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
        'shards': shard_records,
    }
    write_json_atomic(manifest, manifest_path)
    return features, manifest


print('Feature cache helpers ready.')
print('Cache root:', CACHE_DIR)
print('Per-variant train cache size: {:.2f} GiB'.format(
    EXPECTED_TRAIN_ROWS * NUM_BRANCHES * HIDDEN_SIZE * 2 / 2 ** 30
))

In [ ]:
HEAD_DEVICE = 'cuda:0'


class CachedBranchRows(Dataset):
    """Serves rows of a cached [N, 2, 4096] FP16 tensor without copying it."""

    def __init__(self, features, indices, labels=None):
        self.features = features
        self.indices = [int(index) for index in indices]
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        label = int(self.labels[index]) if self.labels is not None else -1
        return self.features[index], label, index


class ControlHead(nn.Module):
    """The adopted V10 readout, unchanged: concat(final CLS, final patch mean) -> 8192 -> 3."""

    def __init__(self):
        super().__init__()
        input_dim = NUM_BRANCHES * HIDDEN_SIZE
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, HEAD_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(HEAD_HIDDEN_DIM, NUM_CLASSES),
        )

    def forward(self, features):
        return self.network(features.flatten(1))


def softmax_numpy(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('Head probabilities contain NaN or Inf.')
    return probabilities


def predict_probabilities(model, features, indices):
    index_list = [int(index) for index in indices]
    loader = DataLoader(
        CachedBranchRows(features, index_list),
        batch_size=PREDICT_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    logits_parts = []
    observed_indices = []
    model.eval()
    with torch.inference_mode():
        for batch_features, _batch_labels, batch_indices in loader:
            logits = model(batch_features.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True))
            if not torch.isfinite(logits).all().item():
                raise RuntimeError('Head logits contain NaN or Inf.')
            logits_parts.append(logits.float().cpu())
            observed_indices.extend(int(index) for index in batch_indices.tolist())
    if observed_indices != index_list:
        raise RuntimeError('Prediction order changed inside the DataLoader.')
    stacked = torch.cat(logits_parts).numpy().astype(np.float64)
    if stacked.shape != (len(index_list), NUM_CLASSES):
        raise RuntimeError('Head prediction matrix has shape {}.'.format(stacked.shape))
    return softmax_numpy(stacked)


def distribution_summary(values):
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        return {'count': 0, 'mean': None, 'std': None, 'q10': None, 'median': None, 'q90': None}
    return {
        'count': int(len(values)),
        'mean': float(values.mean()),
        'std': float(values.std()),
        'q10': float(np.quantile(values, 0.10)),
        'median': float(np.quantile(values, 0.50)),
        'q90': float(np.quantile(values, 0.90)),
    }


def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(labels), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError('Metric input probabilities are incomplete or non-finite.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        labels, predictions, labels=list(range(NUM_CLASSES)),
        target_names=list(CLASS_NAMES), output_dict=True, zero_division=0,
    )
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    confidence = top_two[:, 1]
    margin = top_two[:, 1] - top_two[:, 0]
    correct = predictions == labels
    recyclable_to_organic = int(matrix[0, 2])
    organic_to_recyclable = int(matrix[2, 0])
    return {
        'rows': int(len(labels)),
        'macro_f1': float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)), average='macro')),
        'accuracy': float(accuracy_score(labels, predictions)),
        'log_loss': float(log_loss(labels, probabilities, labels=list(range(NUM_CLASSES)))),
        'per_class': {
            name: {
                'precision': float(report[name]['precision']),
                'recall': float(report[name]['recall']),
                'f1': float(report[name]['f1-score']),
                'support': int(report[name]['support']),
            }
            for name in CLASS_NAMES
        },
        'f1_recyclable': float(report['Recyclable']['f1-score']),
        'f1_electronic': float(report['Electronic']['f1-score']),
        'f1_organic': float(report['Organic']['f1-score']),
        'confusion_matrix': matrix.tolist(),
        'total_errors': int((~correct).sum()),
        'recyclable_to_organic': recyclable_to_organic,
        'organic_to_recyclable': organic_to_recyclable,
        'ro_errors': recyclable_to_organic + organic_to_recyclable,
        'electronic_errors': int(matrix[1].sum() - matrix[1, 1]),
        'high_confidence_errors': int(((~correct) & (confidence >= 0.90)).sum()),
        'confidence_summary': {
            'all': distribution_summary(confidence),
            'correct': distribution_summary(confidence[correct]),
            'errors': distribution_summary(confidence[~correct]),
        },
        'margin_summary': {
            'all': distribution_summary(margin),
            'correct': distribution_summary(margin[correct]),
            'errors': distribution_summary(margin[~correct]),
        },
    }


def predictions_frame(indices, probabilities, variant_name, seed, validation_fold):
    indices = np.asarray(indices, dtype=np.int64)
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    predictions = probabilities.argmax(axis=1)
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    window = train_df.iloc[indices]
    return pd.DataFrame({
        'row_index': indices,
        'image_id': window['image_id'].to_numpy(),
        'label': labels,
        'prediction': predictions,
        'prob_0': probabilities[:, 0],
        'prob_1': probabilities[:, 1],
        'prob_2': probabilities[:, 2],
        'confidence': top_two[:, 1],
        'top_two_margin': top_two[:, 1] - top_two[:, 0],
        'correct': labels == predictions,
        'is_upsampled': window['is_upsampled'].to_numpy(),
        'fold': validation_fold,
        'variant': variant_name,
        'seed': seed,
    })


def class_weights_from(label_values):
    class_counts = np.bincount(label_values, minlength=NUM_CLASSES)
    if (class_counts <= 0).any():
        raise RuntimeError('A training partition is missing a class: {}.'.format(class_counts.tolist()))
    weights = class_counts.sum() / (NUM_CLASSES * class_counts)
    return class_counts, torch.tensor(weights, dtype=torch.float32, device=HEAD_DEVICE)


def run_training_epochs(model, features, train_indices, epochs, seed, criterion, optimizer, scheduler, on_epoch_end):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        CachedBranchRows(features, train_indices, cached_labels),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
    )
    for epoch in range(1, epochs + 1):
        epoch_started = time.perf_counter()
        model.train()
        loss_sum = 0.0
        seen = 0
        learning_rate = optimizer.param_groups[0]['lr']
        for batch_features, batch_labels, _batch_indices in train_loader:
            batch_features = batch_features.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True)
            batch_labels = batch_labels.to(device=HEAD_DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            if not torch.isfinite(loss).item():
                raise RuntimeError('Non-finite loss at epoch {}.'.format(epoch))
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            if not torch.isfinite(gradient_norm).item():
                raise RuntimeError('Non-finite head gradient norm at epoch {}.'.format(epoch))
            optimizer.step()
            loss_sum += loss.item() * len(batch_labels)
            seen += len(batch_labels)
        scheduler.step()
        if on_epoch_end(epoch, loss_sum / seen, learning_rate, time.perf_counter() - epoch_started):
            break
    del train_loader


def train_fold_head(variant_name, features, validation_fold, seed, persist=True, verbose=True):
    """One fold, one seed, the locked head protocol. Identical for both variants."""
    fold_values = cached_folds.numpy()
    train_indices = np.flatnonzero(fold_values != validation_fold)
    valid_indices = np.flatnonzero(fold_values == validation_fold)
    if set(fold_values[valid_indices].tolist()) != {validation_fold}:
        raise RuntimeError('Validation fold membership is invalid.')

    train_label_values = cached_labels[torch.from_numpy(train_indices)].numpy()
    valid_labels = cached_labels[torch.from_numpy(valid_indices)].numpy()
    class_counts, class_weights = class_weights_from(train_label_values)

    seed_everything(seed)
    model = ControlHead().to(HEAD_DEVICE)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    state = {'best_macro_f1': -1.0, 'best_log_loss': float('inf'), 'best_epoch': None, 'best_state': None, 'stale': 0}
    history = []
    training_started = time.perf_counter()

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        valid_probabilities = predict_probabilities(model, features, valid_indices)
        valid_predictions = valid_probabilities.argmax(axis=1)
        valid_macro_f1 = float(
            f1_score(valid_labels, valid_predictions, labels=list(range(NUM_CLASSES)), average='macro')
        )
        valid_log_loss = float(log_loss(valid_labels, valid_probabilities, labels=list(range(NUM_CLASSES))))
        history.append({
            'variant': variant_name, 'validation_fold': validation_fold, 'seed': seed, 'epoch': epoch,
            'train_loss': train_loss, 'valid_macro_f1': valid_macro_f1, 'valid_log_loss': valid_log_loss,
            'learning_rate': learning_rate, 'epoch_seconds': epoch_seconds,
        })
        if verbose:
            print('    {:<22} fold={} seed={} epoch={:02d} loss={:.6f} f1={:.6f} logloss={:.6f}'.format(
                variant_name, validation_fold, seed, epoch, train_loss, valid_macro_f1, valid_log_loss
            ))
        improved = valid_macro_f1 > state['best_macro_f1'] + 1e-8 or (
            abs(valid_macro_f1 - state['best_macro_f1']) <= 1e-8 and valid_log_loss < state['best_log_loss']
        )
        if improved:
            state['best_macro_f1'] = valid_macro_f1
            state['best_log_loss'] = valid_log_loss
            state['best_epoch'] = epoch
            state['best_state'] = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            state['stale'] = 0
        else:
            state['stale'] += 1
        return state['stale'] >= EARLY_STOPPING_PATIENCE

    run_training_epochs(
        model, features, train_indices, MAX_EPOCHS, seed, criterion, optimizer, scheduler, on_epoch_end
    )
    if state['best_state'] is None:
        raise RuntimeError('Head training produced no valid checkpoint.')
    training_seconds = time.perf_counter() - training_started

    model.load_state_dict(state['best_state'], strict=True)
    final_probabilities = predict_probabilities(model, features, valid_indices)
    metrics = metric_bundle(valid_labels, final_probabilities)
    metrics.update({
        'variant': variant_name,
        'validation_fold': validation_fold,
        'seed': seed,
        'best_epoch': state['best_epoch'],
        'parameter_count': parameter_count,
        'training_seconds': training_seconds,
        'train_class_counts': class_counts.tolist(),
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    })

    prediction_table = predictions_frame(valid_indices, final_probabilities, variant_name, seed, validation_fold)
    if persist:
        run_stem = '{}_fold{}_seed{}'.format(variant_name, validation_fold, seed)
        write_csv_atomic(prediction_table, HEADS_DIR / (run_stem + '_predictions.csv'))
        write_csv_atomic(pd.DataFrame(history), HEADS_DIR / (run_stem + '_history.csv'))
        write_json_atomic(metrics, HEADS_DIR / (run_stem + '_metrics.json'))

    del model, criterion, optimizer, scheduler
    state['best_state'] = None
    release_cuda_cache()
    return metrics, prediction_table


def run_five_fold_oof(variant_name, features, seed, persist=True, verbose=True):
    """Full five-fold OOF over all 26,527 rows. No fold ensembling: each row is predicted once."""
    per_fold = []
    frames = []
    for fold in range(N_FOLDS):
        metrics, frame = train_fold_head(variant_name, features, fold, seed, persist=persist, verbose=verbose)
        per_fold.append(metrics)
        frames.append(frame)
        print('  [{}] seed={} fold={} macro_f1={:.10f} errors={} R/O={} elecF1={:.6f} best_epoch={}'.format(
            variant_name, seed, fold, metrics['macro_f1'], metrics['total_errors'],
            metrics['ro_errors'], metrics['f1_electronic'], metrics['best_epoch'],
        ))
    pooled = pd.concat(frames, ignore_index=True).sort_values('row_index').reset_index(drop=True)
    if pooled['row_index'].tolist() != list(range(len(train_df))):
        raise RuntimeError('OOF predictions are incomplete or misaligned.')
    oof = metric_bundle(
        pooled['label'].to_numpy(dtype=np.int64),
        pooled[['prob_0', 'prob_1', 'prob_2']].to_numpy(dtype=np.float64),
    )
    oof['variant'] = variant_name
    oof['seed'] = seed
    oof['per_fold_macro_f1'] = {str(m['validation_fold']): m['macro_f1'] for m in per_fold}
    oof['best_epochs'] = {str(m['validation_fold']): m['best_epoch'] for m in per_fold}
    # Mechanism check: did the change act where the hypothesis says it should?
    upsampled_mask = pooled['is_upsampled'].to_numpy(dtype=bool)
    wrong = (pooled['prediction'].to_numpy() != pooled['label'].to_numpy())
    oof['errors_on_upsampled_rows'] = int((wrong & upsampled_mask).sum())
    oof['errors_on_native_or_downsampled_rows'] = int((wrong & (~upsampled_mask)).sum())
    return per_fold, oof, pooled


probe_head = ControlHead()
HEAD_PARAMETER_COUNT = sum(parameter.numel() for parameter in probe_head.parameters())
with torch.no_grad():
    probe_logits = probe_head(torch.zeros(2, NUM_BRANCHES, HIDDEN_SIZE))
if tuple(probe_logits.shape) != (2, NUM_CLASSES):
    raise RuntimeError('Head produced logits with shape {}.'.format(tuple(probe_logits.shape)))
del probe_head, probe_logits

print('Head parameters       : {:,}'.format(HEAD_PARAMETER_COUNT))
print('V10 head parameters   : 4,212,739 (must match)')
if HEAD_PARAMETER_COUNT != 4212739:
    raise RuntimeError(
        'Head parameter count {:,} does not match the adopted V10 head (4,212,739). The head is not the '
        'champion head, so this would no longer be a single-change experiment.'.format(HEAD_PARAMETER_COUNT)
    )
print('Head architecture     : LayerNorm(8192) -> Linear(512) -> GELU -> Dropout(0.4) -> Linear(3)')

In [ ]:
TRAIN_FEATURES = {}
TRAIN_CACHE_MANIFEST = {}
PER_FOLD_METRICS = {}
OOF_METRICS = {}
OOF_PREDICTIONS = {}

for variant_name in VARIANT_ORDER:
    print()
    print('=' * 78)
    print('VARIANT {}  (interpolation = {})   [t+{:.0f} min]'.format(
        variant_name, VARIANT_INTERPOLATION[variant_name], elapsed_minutes()
    ))
    print('=' * 78)

    features, manifest = build_feature_cache(variant_name, 'train', train_df)
    TRAIN_FEATURES[variant_name] = features
    TRAIN_CACHE_MANIFEST[variant_name] = manifest
    release_cuda_cache()
    print('Cache ready: {} rows, {:.2f} GiB, written {}, reused {}'.format(
        manifest['rows'], manifest['feature_bytes'] / 2 ** 30,
        manifest['rows_written_this_session'], manifest['rows_reused_this_session'],
    ))
    print('Cache signature:', manifest['cache_signature'])

    print('Training five folds at seed {}...'.format(SEED))
    per_fold, oof, pooled = run_five_fold_oof(variant_name, features, SEED, persist=True, verbose=False)
    PER_FOLD_METRICS[variant_name] = per_fold
    OOF_METRICS[variant_name] = oof
    OOF_PREDICTIONS[variant_name] = pooled

    write_csv_atomic(pooled, ARTIFACT_DIR / 'oof_predictions_{}.csv'.format(variant_name))
    write_npy_atomic(
        pooled[['prob_0', 'prob_1', 'prob_2']].to_numpy(dtype=np.float32),
        ARTIFACT_DIR / 'oof_probabilities_{}.npy'.format(variant_name),
    )
    write_json_atomic(oof, ARTIFACT_DIR / 'oof_metrics_{}.json'.format(variant_name))

    print()
    print('  FULL 5-FOLD OOF [{}] seed {}'.format(variant_name, SEED))
    print('    macro F1        : {:.10f}'.format(oof['macro_f1']))
    print('    total errors    : {}'.format(oof['total_errors']))
    print('    R/O errors      : {}  (R->O {}, O->R {})'.format(
        oof['ro_errors'], oof['recyclable_to_organic'], oof['organic_to_recyclable']
    ))
    print('    Electronic err  : {}   Electronic F1: {:.10f}'.format(
        oof['electronic_errors'], oof['f1_electronic']
    ))
    print('    per-class F1    : R {:.10f} | E {:.10f} | O {:.10f}'.format(
        oof['f1_recyclable'], oof['f1_electronic'], oof['f1_organic']
    ))
    print('    log loss        : {:.10f}'.format(oof['log_loss']))
    print('    errors on upsampled rows        : {}'.format(oof['errors_on_upsampled_rows']))
    print('    errors on native/downsampled    : {}'.format(oof['errors_on_native_or_downsampled_rows']))

per_fold_table = pd.DataFrame([
    {
        'variant': metrics['variant'],
        'seed': metrics['seed'],
        'validation_fold': metrics['validation_fold'],
        'best_epoch': metrics['best_epoch'],
        'rows': metrics['rows'],
        'macro_f1': metrics['macro_f1'],
        'accuracy': metrics['accuracy'],
        'log_loss': metrics['log_loss'],
        'total_errors': metrics['total_errors'],
        'recyclable_to_organic': metrics['recyclable_to_organic'],
        'organic_to_recyclable': metrics['organic_to_recyclable'],
        'ro_errors': metrics['ro_errors'],
        'electronic_errors': metrics['electronic_errors'],
        'f1_electronic': metrics['f1_electronic'],
        'training_seconds': metrics['training_seconds'],
    }
    for variant_name in VARIANT_ORDER
    for metrics in PER_FOLD_METRICS[variant_name]
]).sort_values(['variant', 'validation_fold']).reset_index(drop=True)
write_csv_atomic(per_fold_table, ARTIFACT_DIR / 'per_fold_seed42.csv')

print()
print('=' * 78)
print('PER-FOLD TABLE (seed {})'.format(SEED))
print('=' * 78)
display(per_fold_table[[
    'variant', 'validation_fold', 'best_epoch', 'macro_f1', 'total_errors', 'ro_errors',
    'electronic_errors', 'f1_electronic',
]])
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
# ---------------------------------------------------------------------------
# PIPELINE INTEGRITY: does the in-session control reproduce the recorded champion?
# This runs BEFORE the adoption gate on purpose. If the control does not land on
# the V10 number, the whole run is uninterpretable and adoption is refused.
# ---------------------------------------------------------------------------
control_oof = OOF_METRICS[CONTROL_VARIANT]
control_macro_f1_delta = control_oof['macro_f1'] - V10_REFERENCE['oof_macro_f1']

CONTROL_REPRODUCTION = {
    'v10_recorded_macro_f1': V10_REFERENCE['oof_macro_f1'],
    'v13_in_session_control_macro_f1': control_oof['macro_f1'],
    'macro_f1_delta': control_macro_f1_delta,
    'tolerance': CONTROL_REPRODUCTION_TOLERANCE,
    'within_tolerance': bool(abs(control_macro_f1_delta) <= CONTROL_REPRODUCTION_TOLERANCE),
    'v10_recorded_total_errors': V10_REFERENCE['oof_total_errors'],
    'v13_control_total_errors': control_oof['total_errors'],
    'total_error_delta': control_oof['total_errors'] - V10_REFERENCE['oof_total_errors'],
    'v10_recorded_ro_errors': V10_REFERENCE['oof_ro_errors'],
    'v13_control_ro_errors': control_oof['ro_errors'],
    'ro_error_delta': control_oof['ro_errors'] - V10_REFERENCE['oof_ro_errors'],
    'model_revision_matches_v10': RESOLVED_REVISION == 'b40f6c827cb02b13c7a23a76cf598489bb0f94e5',
    'known_non_identity_sources': [
        'V10 pooled the final block out of forward_intermediates(norm=True); V13 uses forward_features. '
        'V10 proved these are numerically equivalent in its own preflight, but reduction order can still '
        'move the last bits of an FP16 tensor.',
        'different session, possibly different timm/torch build and a different resolved model revision',
    ],
}
PIPELINE_INTEGRITY_OK = CONTROL_REPRODUCTION['within_tolerance']
write_json_atomic(CONTROL_REPRODUCTION, ARTIFACT_DIR / 'control_reproduction.json')

print('=' * 78)
print('PIPELINE INTEGRITY CHECK')
print('=' * 78)
print('V10 recorded OOF macro F1 : {:.10f}  (errors {}, R/O {})'.format(
    V10_REFERENCE['oof_macro_f1'], V10_REFERENCE['oof_total_errors'], V10_REFERENCE['oof_ro_errors']
))
print('V13 in-session control    : {:.10f}  (errors {}, R/O {})'.format(
    control_oof['macro_f1'], control_oof['total_errors'], control_oof['ro_errors']
))
print('delta                     : {:+.10f}  (tolerance +/-{:.4f})'.format(
    control_macro_f1_delta, CONTROL_REPRODUCTION_TOLERANCE
))
print('PIPELINE INTEGRITY        :', 'PASS' if PIPELINE_INTEGRITY_OK else 'FAIL')
if not PIPELINE_INTEGRITY_OK:
    print()
    print('!' * 78)
    print('!! The control did not reproduce the champion. Treat every number below as       !!')
    print('!! UNINTERPRETABLE. Adoption is refused automatically and the submission will be !!')
    print('!! built from the control recipe, which is what V10 already used.                !!')
    print('!' * 78)

# ---------------------------------------------------------------------------
# SEED STABILITY: confirmation only. Seed 42 alone drives the decision.
# ---------------------------------------------------------------------------
print()
print('=' * 78)
print('SEED STABILITY (confirmation, never selection)   [t+{:.0f} min]'.format(elapsed_minutes()))
print('=' * 78)

SEED_OOF = {variant_name: {SEED: OOF_METRICS[variant_name]} for variant_name in VARIANT_ORDER}
for stability_seed in STABILITY_SEEDS:
    for variant_name in VARIANT_ORDER:
        _, oof, _ = run_five_fold_oof(
            variant_name, TRAIN_FEATURES[variant_name], stability_seed, persist=False, verbose=False
        )
        SEED_OOF[variant_name][stability_seed] = oof

ALL_SEEDS = (SEED,) + tuple(STABILITY_SEEDS)
seed_rows = []
for seed_value in ALL_SEEDS:
    control_seed_oof = SEED_OOF[CONTROL_VARIANT][seed_value]
    native_seed_oof = SEED_OOF[NATIVE_VARIANT][seed_value]
    seed_rows.append({
        'seed': seed_value,
        'control_macro_f1': control_seed_oof['macro_f1'],
        'native_macro_f1': native_seed_oof['macro_f1'],
        'macro_f1_gain': native_seed_oof['macro_f1'] - control_seed_oof['macro_f1'],
        'control_total_errors': control_seed_oof['total_errors'],
        'native_total_errors': native_seed_oof['total_errors'],
        'control_ro_errors': control_seed_oof['ro_errors'],
        'native_ro_errors': native_seed_oof['ro_errors'],
        'control_f1_electronic': control_seed_oof['f1_electronic'],
        'native_f1_electronic': native_seed_oof['f1_electronic'],
    })
seed_stability_table = pd.DataFrame(seed_rows)
write_csv_atomic(seed_stability_table, ARTIFACT_DIR / 'seed_stability.csv')

SEED_GAINS = {str(row['seed']): float(row['macro_f1_gain']) for row in seed_rows}
gain_signs = {1 if value > 0 else (-1 if value < 0 else 0) for value in SEED_GAINS.values()}
SEED_SIGN_CONSISTENT = len(gain_signs) == 1 and 0 not in gain_signs
CONTROL_SEED_SPREAD = float(np.std([row['control_macro_f1'] for row in seed_rows]))

print()
display(seed_stability_table)
print('Gain by seed          :', {key: round(value, 8) for key, value in SEED_GAINS.items()})
print('Sign consistent       :', SEED_SIGN_CONSISTENT)
print('Control seed-to-seed std of OOF macro F1: {:.8f}'.format(CONTROL_SEED_SPREAD))
print('  -> read the seed-42 gain against this number. A gain smaller than the spread is noise,')
print('     no matter what the gate says.')
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
control_oof = OOF_METRICS[CONTROL_VARIANT]
native_oof = OOF_METRICS[NATIVE_VARIANT]

OOF_MACRO_F1_GAIN = native_oof['macro_f1'] - control_oof['macro_f1']
ELECTRONIC_F1_DROP = control_oof['f1_electronic'] - native_oof['f1_electronic']

fold_macro_f1 = {
    variant_name: {
        int(metrics['validation_fold']): float(metrics['macro_f1'])
        for metrics in PER_FOLD_METRICS[variant_name]
    }
    for variant_name in VARIANT_ORDER
}
ASSESSMENT_FOLD_GAINS = {
    str(fold): fold_macro_f1[NATIVE_VARIANT][fold] - fold_macro_f1[CONTROL_VARIANT][fold]
    for fold in ASSESSMENT_FOLDS
}
DEVELOPMENT_FOLD_GAIN = (
    fold_macro_f1[NATIVE_VARIANT][DEVELOPMENT_FOLD] - fold_macro_f1[CONTROL_VARIANT][DEVELOPMENT_FOLD]
)
assessment_gains = list(ASSESSMENT_FOLD_GAINS.values())
FOLDS_WON = sum(1 for gain in assessment_gains if gain > 0)
positive_gain_total = sum(gain for gain in assessment_gains if gain > 0)
GAIN_CONCENTRATION = (max(assessment_gains) / positive_gain_total) if positive_gain_total > 0 else 1.0

ADOPTION_GATE_CHECKS = {
    'pipeline_integrity_control_reproduces_v10': bool(PIPELINE_INTEGRITY_OK),
    'oof_macro_f1_gain_at_least_threshold': bool(OOF_MACRO_F1_GAIN >= ADOPTION_MIN_OOF_MACRO_F1_GAIN),
    'oof_total_errors_decrease': bool(native_oof['total_errors'] < control_oof['total_errors']),
    'oof_ro_errors_decrease': bool(native_oof['ro_errors'] < control_oof['ro_errors']),
    'wins_at_least_three_of_four_assessment_folds': bool(FOLDS_WON >= ADOPTION_MIN_ASSESSMENT_FOLDS_WON),
    'electronic_f1_stable': bool(ELECTRONIC_F1_DROP <= ADOPTION_MAX_ELECTRONIC_F1_DROP),
    'improvement_not_concentrated_in_one_fold': bool(GAIN_CONCENTRATION <= ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE),
    'gain_sign_consistent_across_seeds': bool(SEED_SIGN_CONSISTENT),
    'no_test_access_during_selection': bool(
        not TEST_ACCESS_STATE['test_images_accessed'] and not TEST_ACCESS_STATE['manual_test_labels_accessed']
    ),
}
NATIVE_ADOPTED = all(ADOPTION_GATE_CHECKS.values())
SELECTED_VARIANT = NATIVE_VARIANT if NATIVE_ADOPTED else CONTROL_VARIANT
SELECTION_REASON = (
    'bicubic_model_native cleared every locked gate; the resize kernel was the only change'
    if NATIVE_ADOPTED
    else 'bicubic_model_native failed at least one locked gate; the V10 control recipe stands as champion'
)

STRONG_EVIDENCE = bool(OOF_MACRO_F1_GAIN >= STRONG_EVIDENCE_MACRO_F1_GAIN)
GAIN_EXCEEDS_SEED_NOISE = bool(abs(OOF_MACRO_F1_GAIN) > 2.0 * CONTROL_SEED_SPREAD)

SELECTION_DECISION = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'selected_variant': SELECTED_VARIANT,
    'selection_reason': SELECTION_REASON,
    'selection_basis': 'train_only_cross_validation',
    'native_adopted': NATIVE_ADOPTED,
    'adoption_gate_checks': ADOPTION_GATE_CHECKS,
    'oof_macro_f1_gain': OOF_MACRO_F1_GAIN,
    'electronic_f1_drop': ELECTRONIC_F1_DROP,
    'development_fold_gain': DEVELOPMENT_FOLD_GAIN,
    'assessment_fold_gains': ASSESSMENT_FOLD_GAINS,
    'assessment_folds_won': FOLDS_WON,
    'largest_single_fold_gain_share': GAIN_CONCENTRATION,
    'seed_gains': SEED_GAINS,
    'control_seed_spread': CONTROL_SEED_SPREAD,
    'gain_exceeds_twice_seed_noise': GAIN_EXCEEDS_SEED_NOISE,
    'strong_evidence_threshold_met': STRONG_EVIDENCE,
    'out_of_fold_metrics': {variant_name: OOF_METRICS[variant_name] for variant_name in VARIANT_ORDER},
    'control_reproduction': CONTROL_REPRODUCTION,
    'per_fold_macro_f1': {variant_name: fold_macro_f1[variant_name] for variant_name in VARIANT_ORDER},
    'blending_used': False,
    'fold_ensembling_used': False,
    'seed_ensembling_used': False,
    'threshold_tuning_used': False,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'preflight_signature': PREFLIGHT_SIGNATURE,
    'train_cache_signatures': {
        variant_name: TRAIN_CACHE_MANIFEST[variant_name]['cache_signature'] for variant_name in VARIANT_ORDER
    },
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
SELECTION_DECISION_SIGNATURE = sha256_json(SELECTION_DECISION)
write_json_atomic(
    {'selection_decision': SELECTION_DECISION, 'selection_decision_signature': SELECTION_DECISION_SIGNATURE},
    SELECTION_DECISION_PATH,
)

print('=' * 78)
print('ADOPTION GATE   (locked before any V13 number existed)')
print('=' * 78)
print()
print('{:<26} {:>18} {:>18}'.format('metric', 'control(bilinear)', 'native(bicubic)'))
print('-' * 66)
print('{:<26} {:>18.10f} {:>18.10f}'.format('OOF macro F1', control_oof['macro_f1'], native_oof['macro_f1']))
print('{:<26} {:>18d} {:>18d}'.format('OOF total errors', control_oof['total_errors'], native_oof['total_errors']))
print('{:<26} {:>18d} {:>18d}'.format('OOF R/O errors', control_oof['ro_errors'], native_oof['ro_errors']))
print('{:<26} {:>18d} {:>18d}'.format('  R->O', control_oof['recyclable_to_organic'], native_oof['recyclable_to_organic']))
print('{:<26} {:>18d} {:>18d}'.format('  O->R', control_oof['organic_to_recyclable'], native_oof['organic_to_recyclable']))
print('{:<26} {:>18d} {:>18d}'.format('OOF Electronic errors', control_oof['electronic_errors'], native_oof['electronic_errors']))
print('{:<26} {:>18.10f} {:>18.10f}'.format('Electronic F1', control_oof['f1_electronic'], native_oof['f1_electronic']))
print('{:<26} {:>18.10f} {:>18.10f}'.format('Recyclable F1', control_oof['f1_recyclable'], native_oof['f1_recyclable']))
print('{:<26} {:>18.10f} {:>18.10f}'.format('Organic F1', control_oof['f1_organic'], native_oof['f1_organic']))
print('{:<26} {:>18.10f} {:>18.10f}'.format('log loss', control_oof['log_loss'], native_oof['log_loss']))
print('{:<26} {:>18d} {:>18d}'.format('high-confidence errors', control_oof['high_confidence_errors'], native_oof['high_confidence_errors']))
print('{:<26} {:>18d} {:>18d}'.format('errors on upsampled', control_oof['errors_on_upsampled_rows'], native_oof['errors_on_upsampled_rows']))
print('{:<26} {:>18d} {:>18d}'.format('errors on native-res', control_oof['errors_on_native_or_downsampled_rows'], native_oof['errors_on_native_or_downsampled_rows']))
print()
print('OOF macro F1 gain     : {:+.10f}   (gate {:+.4f})'.format(OOF_MACRO_F1_GAIN, ADOPTION_MIN_OOF_MACRO_F1_GAIN))
print('Fold 0 gain           : {:+.10f}'.format(DEVELOPMENT_FOLD_GAIN))
print('Assessment fold gains : {}'.format({key: round(value, 8) for key, value in ASSESSMENT_FOLD_GAINS.items()}))
print('Assessment folds won  : {} of 4   (gate >= {})'.format(FOLDS_WON, ADOPTION_MIN_ASSESSMENT_FOLDS_WON))
print('Largest fold share    : {:.4f}   (gate <= {:.2f})'.format(GAIN_CONCENTRATION, ADOPTION_MAX_SINGLE_FOLD_GAIN_SHARE))
print('Electronic F1 drop    : {:+.10f}   (gate <= {:.4f})'.format(ELECTRONIC_F1_DROP, ADOPTION_MAX_ELECTRONIC_F1_DROP))
print()
for check_name, passed in ADOPTION_GATE_CHECKS.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
print()
print('Gain exceeds 2x seed noise : {}  (|gain| {:.8f} vs 2*std {:.8f})'.format(
    GAIN_EXCEEDS_SEED_NOISE, abs(OOF_MACRO_F1_GAIN), 2.0 * CONTROL_SEED_SPREAD
))
print('Strong-evidence threshold  : {}  (>= {:+.4f})'.format(STRONG_EVIDENCE, STRONG_EVIDENCE_MACRO_F1_GAIN))
print()
print('=' * 78)
print('DECISION: {}'.format('ADOPT bicubic_model_native' if NATIVE_ADOPTED else 'REJECT bicubic_model_native'))
print('SELECTED VARIANT FOR THE SUBMISSION: {}'.format(SELECTED_VARIANT))
print('REASON: {}'.format(SELECTION_REASON))
print('=' * 78)
print('selection_decision.json written BEFORE any test image is opened.')
print('Signature:', SELECTION_DECISION_SIGNATURE)

In [ ]:
def train_full_head(variant_name, features, epochs):
    """Retrain one head from scratch on every official raw training row. No validation, no early stopping."""
    all_indices = np.arange(len(train_df), dtype=np.int64)
    class_counts, class_weights = class_weights_from(cached_labels.numpy())

    seed_everything(SEED)
    model = ControlHead().to(HEAD_DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    # T_max stays at MAX_EPOCHS so the learning-rate trajectory matches the cross-validation runs.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    history = []
    started = time.perf_counter()

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        history.append({
            'stage': 'full_training', 'variant': variant_name, 'seed': SEED, 'epoch': epoch,
            'train_loss': train_loss, 'learning_rate': learning_rate, 'epoch_seconds': epoch_seconds,
        })
        print('  full_training {} epoch={:02d}/{:02d} loss={:.6f} lr={:.6e}'.format(
            variant_name, epoch, epochs, train_loss, learning_rate
        ))
        return False

    run_training_epochs(
        model, features, all_indices, epochs, SEED, criterion, optimizer, scheduler, on_epoch_end
    )
    metadata = {
        'schema_version': 1,
        'revision_marker': REVISION_MARKER,
        'variant': variant_name,
        'interpolation': VARIANT_INTERPOLATION[variant_name],
        'trained_on': 'all {} official raw training rows'.format(len(train_df)),
        'epochs': epochs,
        'epoch_selection': 'median best epoch across the selected variant cross-validation folds',
        'early_stopping_used': False,
        'seed': SEED,
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'batch_size': HEAD_BATCH_SIZE,
        'label_smoothing': LABEL_SMOOTHING,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
        'scheduler_t_max': MAX_EPOCHS,
        'train_class_counts': class_counts.tolist(),
        'parameter_count': sum(parameter.numel() for parameter in model.parameters()),
        'training_seconds': time.perf_counter() - started,
        'cache_signature': TRAIN_CACHE_MANIFEST[variant_name]['cache_signature'],
        'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    }
    return model, metadata, pd.DataFrame(history)


selected_best_epochs = [
    int(metrics['best_epoch']) for metrics in PER_FOLD_METRICS[SELECTED_VARIANT]
]
if len(selected_best_epochs) != N_FOLDS:
    raise RuntimeError('Expected {} best-epoch values for the selected variant.'.format(N_FOLDS))
FULL_TRAINING_EPOCHS = int(round(float(np.median(selected_best_epochs))))
if FULL_TRAINING_EPOCHS < 1 or FULL_TRAINING_EPOCHS > MAX_EPOCHS:
    raise RuntimeError('Derived full-training epoch count {} is out of range.'.format(FULL_TRAINING_EPOCHS))

print('Selected variant     :', SELECTED_VARIANT)
print('Per-fold best epochs :', selected_best_epochs)
print('Median -> full epochs:', FULL_TRAINING_EPOCHS, '(V10 used 5)')
print()

FINAL_HEAD, FINAL_HEAD_METADATA, final_training_history = train_full_head(
    SELECTED_VARIANT, TRAIN_FEATURES[SELECTED_VARIANT], FULL_TRAINING_EPOCHS
)
FINAL_HEAD_PATH = ARTIFACT_DIR / 'final_head.pt'
atomic_torch_save(
    {'metadata': FINAL_HEAD_METADATA, 'state_dict': {
        name: tensor.detach().cpu().clone() for name, tensor in FINAL_HEAD.state_dict().items()
    }},
    FINAL_HEAD_PATH,
)
write_csv_atomic(final_training_history, ARTIFACT_DIR / 'final_training_history.csv')
write_json_atomic(FINAL_HEAD_METADATA, ARTIFACT_DIR / 'final_head_metadata.json')

print()
print('Final head trained on all {} rows for {} epochs and saved to {}'.format(
    len(train_df), FULL_TRAINING_EPOCHS, FINAL_HEAD_PATH
))
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
persisted_decision = read_json(SELECTION_DECISION_PATH) if SELECTION_DECISION_PATH.is_file() else {}

TEST_UNLOCK_CHECKS = {
    'selection_decision_file_exists': bool(persisted_decision),
    'selection_decision_signature_matches': (
        persisted_decision.get('selection_decision_signature') == SELECTION_DECISION_SIGNATURE
    ),
    'both_train_caches_complete': all(
        TRAIN_CACHE_MANIFEST[name].get('complete') is True
        and TRAIN_CACHE_MANIFEST[name]['rows'] == EXPECTED_TRAIN_ROWS
        for name in VARIANT_ORDER
    ),
    'five_fold_oof_complete_for_both_variants': all(
        OOF_METRICS[name]['rows'] == EXPECTED_TRAIN_ROWS for name in VARIANT_ORDER
    ),
    'final_head_exists': FINAL_HEAD_PATH.is_file(),
    # Read from the persisted decision, not live state, so rerunning this cell stays idempotent.
    'test_untouched_during_selection': (
        persisted_decision.get('selection_decision', {}).get('test_images_accessed') is False
    ),
    'manual_test_labels_untouched': not TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
failed_unlock_checks = [name for name, passed in TEST_UNLOCK_CHECKS.items() if not passed]
if failed_unlock_checks:
    raise RuntimeError(
        'Test isolation cannot be released. Failed prerequisites: {}. '
        'Rerun the train-only sections until every prerequisite passes.'.format(failed_unlock_checks)
    )

TEST_ACCESS_STATE['test_images_accessed'] = True
print('Test isolation released. All train-only prerequisites passed:')
for check_name in TEST_UNLOCK_CHECKS:
    print('  [PASS]', check_name)
print()


def valid_test_dir(candidate):
    try:
        entries = list(candidate.iterdir())
    except OSError:
        return False
    if len(entries) != EXPECTED_TEST_ROWS:
        return False
    return all(
        entry.is_file() and not entry.is_symlink() and entry.suffix.lower() in ALLOWED_IMAGE_SUFFIXES
        for entry in entries
    )


def resolve_test_dir():
    candidates = [Path.cwd() / 'data' / 'test', KAGGLE_WORKING / 'data' / 'test']
    candidates.extend(discover_named_dirs(KAGGLE_INPUT, 'test', max_depth=3))
    matches = {candidate.resolve() for candidate in candidates if candidate.is_dir() and valid_test_dir(candidate)}
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(path) for path in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError(
            'Expected exactly one official test directory holding {} images; found {}:'.format(
                EXPECTED_TEST_ROWS, len(matches)
            ) + '\n' + rendered
        )
    return matches.pop()


def resolve_submission_template():
    matches = {}
    for root in (Path.cwd() / 'data', KAGGLE_WORKING / 'data', KAGGLE_INPUT):
        if not root.is_dir():
            continue
        for candidate in root.rglob('*.csv'):
            try:
                frame = pd.read_csv(candidate)
            except Exception:
                continue
            if list(frame.columns) == SUBMISSION_COLUMNS and len(frame) == EXPECTED_TEST_ROWS:
                matches[str(candidate.resolve())] = frame
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(path) for path in sorted(matches)) or '  (none)'
        raise RuntimeError(
            'Expected exactly one official submission template with columns {} and {} rows; found {}:'.format(
                SUBMISSION_COLUMNS, EXPECTED_TEST_ROWS, len(matches)
            ) + '\n' + rendered
        )
    path = next(iter(matches))
    return Path(path), matches[path]


TEST_DIR = resolve_test_dir()
TEMPLATE_PATH, submission_template = resolve_submission_template()

stem_to_path = {}
for path in sorted(TEST_DIR.iterdir(), key=lambda item: item.name):
    if path.stem in stem_to_path:
        raise RuntimeError('Duplicate test image stem: {}'.format(path.stem))
    stem_to_path[path.stem] = path

template_ids = submission_template['id'].tolist()
if len(set(template_ids)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('The official template contains duplicate ids.')
ordered_test_paths = []
for raw_id in template_ids:
    key = Path(str(raw_id)).stem
    if key not in stem_to_path:
        raise RuntimeError('Template id {!r} has no matching image file in {}.'.format(raw_id, TEST_DIR))
    ordered_test_paths.append(str(stem_to_path[key].resolve()))
if len(set(ordered_test_paths)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('Template ids did not map to {} distinct test images.'.format(EXPECTED_TEST_ROWS))

test_df = pd.DataFrame({
    'image_id': [str(raw_id) for raw_id in template_ids],
    'path': ordered_test_paths,
})
TEST_ORDER_SHA256 = sha256_json(test_df['image_id'].tolist())

print('Official test directory  :', TEST_DIR)
print('Official template        :', TEMPLATE_PATH)
print('Test rows in template    :', len(test_df))
print('Template row-order sha256:', TEST_ORDER_SHA256)
print()

# Test features are extracted with the SELECTED variant only, through the same code path as train.
test_features, TEST_CACHE_MANIFEST = build_feature_cache(SELECTED_VARIANT, 'test', test_df)
if tuple(test_features.shape) != (EXPECTED_TEST_ROWS, NUM_BRANCHES, HIDDEN_SIZE):
    raise RuntimeError('Test feature tensor has shape {}.'.format(tuple(test_features.shape)))

train_identity = dict(TRAIN_CACHE_MANIFEST[SELECTED_VARIANT]['cache_identity'])
test_identity = dict(TEST_CACHE_MANIFEST['cache_identity'])
shared_keys = set(train_identity) & set(test_identity) - {'split', 'rows', 'image_ids_sha256', 'labels_sha256', 'folds_sha256'}
if {key: train_identity[key] for key in shared_keys} != {key: test_identity[key] for key in shared_keys}:
    raise RuntimeError('The train and test caches were not produced under the same frozen-backbone contract.')
release_cuda_cache()

TEST_INFERENCE_STARTED_AT = time.perf_counter()
test_probabilities = predict_probabilities(FINAL_HEAD, test_features, np.arange(len(test_df), dtype=np.int64))
TEST_INFERENCE_SECONDS = time.perf_counter() - TEST_INFERENCE_STARTED_AT
if test_probabilities.shape != (EXPECTED_TEST_ROWS, NUM_CLASSES) or not np.isfinite(test_probabilities).all():
    raise RuntimeError('Test probability matrix is incomplete or non-finite.')
write_npy_atomic(test_probabilities.astype(np.float32), ARTIFACT_DIR / 'test_probabilities.npy')

test_predictions = test_probabilities.argmax(axis=1).astype(np.int64)
submission = pd.DataFrame({'id': submission_template['id'].to_numpy(), 'predicted': test_predictions})

SUBMISSION_VALIDATION = {
    'row_count_is_expected': bool(len(submission) == EXPECTED_TEST_ROWS),
    'columns_are_exact': bool(list(submission.columns) == SUBMISSION_COLUMNS),
    'no_missing_values': bool(not submission.isna().any().any()),
    'ids_are_unique': bool(submission['id'].is_unique),
    'predictions_in_class_codes': bool(submission['predicted'].isin([0, 1, 2]).all()),
    'row_order_matches_template': bool(submission['id'].tolist() == submission_template['id'].tolist()),
}
failed_validation = [name for name, passed in SUBMISSION_VALIDATION.items() if not passed]
if failed_validation:
    raise RuntimeError('Submission validation failed: {}.'.format(failed_validation))

SUBMISSION_PATH = ARTIFACT_DIR / 'submission_{}.csv'.format(sanitize_for_filename(TEAM_NAME))
write_csv_atomic(submission, SUBMISSION_PATH)

PREDICTED_DISTRIBUTION = {
    str(class_code): int((test_predictions == class_code).sum()) for class_code in range(NUM_CLASSES)
}

print()
print('Test inference: 1 forward pass, argmax over a single probability matrix.')
print('  no TTA, no hflip, no multi-crop, no fold averaging, no blending, no manual correction.')
print('Predicted class distribution :', PREDICTED_DISTRIBUTION)
print('V10 test distribution        :', V10_REFERENCE['test_predicted_class_distribution'])
print('  (diagnostic only. This comparison never influenced training, selection or the predictions.)')
print('Mean max probability         : {:.10f}'.format(float(test_probabilities.max(axis=1).mean())))
print('Submission written           :', SUBMISSION_PATH)
for check_name, passed in SUBMISSION_VALIDATION.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
if TEAM_NAME == 'CHANGE_ME':
    print()
    print('!' * 78)
    print('!! TEAM_NAME was never set, so the file is named submission_CHANGE_ME.csv.       !!')
    print('!! Rename it to submission_<YourTeamName>.csv before uploading.                  !!')
    print('!' * 78)

In [ ]:
# ---------------------------------------------------------------------------
# REPRODUCTION CHECK: rebuild the head from the persisted checkpoint and re-infer.
# V10 precedent: max_abs_probability_difference = 2.98e-08 with identical argmax.
# ---------------------------------------------------------------------------
reloaded_payload = torch.load(FINAL_HEAD_PATH, map_location='cpu', weights_only=False)
reloaded_head = ControlHead().to(HEAD_DEVICE)
reloaded_head.load_state_dict(reloaded_payload['state_dict'], strict=True)
reproduced_probabilities = predict_probabilities(
    reloaded_head, test_features, np.arange(len(test_df), dtype=np.int64)
)
MAX_ABS_PROBABILITY_DIFFERENCE = float(np.abs(reproduced_probabilities - test_probabilities).max())
PREDICTIONS_IDENTICAL = bool(np.array_equal(reproduced_probabilities.argmax(axis=1), test_predictions))

CHECKPOINT_REPRODUCTION = {
    'checkpoint': str(FINAL_HEAD_PATH),
    'max_abs_probability_difference': MAX_ABS_PROBABILITY_DIFFERENCE,
    'tolerance': RELOAD_TOLERANCE,
    'within_tolerance': bool(MAX_ABS_PROBABILITY_DIFFERENCE <= RELOAD_TOLERANCE),
    'identical_argmax': PREDICTIONS_IDENTICAL,
    'v10_precedent_max_abs_probability_difference': 2.98e-08,
}
write_json_atomic(CHECKPOINT_REPRODUCTION, ARTIFACT_DIR / 'checkpoint_reproduction_check.json')
if not (CHECKPOINT_REPRODUCTION['within_tolerance'] and PREDICTIONS_IDENTICAL):
    raise RuntimeError(
        'Checkpoint reproduction failed: max abs probability difference {:.3e}, identical argmax {}. '
        'Do not submit this file.'.format(MAX_ABS_PROBABILITY_DIFFERENCE, PREDICTIONS_IDENTICAL)
    )
del reloaded_head, reloaded_payload
release_cuda_cache()

# ---------------------------------------------------------------------------
# ERROR ANALYSIS ON THE SELECTED VARIANT (train OOF only, never test)
# ---------------------------------------------------------------------------
selected_oof = OOF_METRICS[SELECTED_VARIANT]
selected_pooled = OOF_PREDICTIONS[SELECTED_VARIANT]
wrong_rows = selected_pooled[selected_pooled['prediction'] != selected_pooled['label']].copy()
ro_rows = wrong_rows[
    ((wrong_rows['label'] == 0) & (wrong_rows['prediction'] == 2))
    | ((wrong_rows['label'] == 2) & (wrong_rows['prediction'] == 0))
]
write_csv_atomic(
    wrong_rows.sort_values('confidence', ascending=False),
    ARTIFACT_DIR / 'oof_errors_{}.csv'.format(SELECTED_VARIANT),
)
write_csv_atomic(
    ro_rows.sort_values('confidence', ascending=False),
    ARTIFACT_DIR / 'oof_recyclable_organic_errors_{}.csv'.format(SELECTED_VARIANT),
)

ERROR_ANALYSIS = {
    'selected_variant': SELECTED_VARIANT,
    'oof_total_errors': selected_oof['total_errors'],
    'oof_ro_errors': selected_oof['ro_errors'],
    'recyclable_organic_share_of_errors': (
        selected_oof['ro_errors'] / selected_oof['total_errors'] if selected_oof['total_errors'] else 0.0
    ),
    'oof_electronic_errors': selected_oof['electronic_errors'],
    'oof_high_confidence_errors': selected_oof['high_confidence_errors'],
    'errors_on_upsampled_rows': selected_oof['errors_on_upsampled_rows'],
    'errors_on_native_or_downsampled_rows': selected_oof['errors_on_native_or_downsampled_rows'],
    'upsampled_row_count': UPSAMPLED_ROWS,
    'historical_v10_oof_total_errors': V10_REFERENCE['oof_total_errors'],
    'historical_v10_oof_ro_errors': V10_REFERENCE['oof_ro_errors'],
    'test_images_inspected_for_model_decisions': False,
}
write_json_atomic(ERROR_ANALYSIS, ARTIFACT_DIR / 'error_analysis.json')

TEST_INFERENCE_MANIFEST = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'selected_variant': SELECTED_VARIANT,
    'interpolation': VARIANT_INTERPOLATION[SELECTED_VARIANT],
    'selection_reason': SELECTION_REASON,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'decision_rule': 'argmax over the single probability matrix',
    'inference_passes': 1,
    'inference_seconds': TEST_INFERENCE_SECONDS,
    'final_head_checkpoint': str(FINAL_HEAD_PATH),
    'full_training_epochs': FULL_TRAINING_EPOCHS,
    'fold_or_seed_ensembling': False,
    'probability_blending': False,
    'test_time_augmentation': False,
    'horizontal_flip_inference': False,
    'multi_crop_inference': False,
    'manual_prediction_correction': False,
    'leaderboard_feedback_used': False,
    'previous_submissions_loaded': False,
    'manual_test_labels_accessed': False,
    'predicted_class_distribution': PREDICTED_DISTRIBUTION,
    'mean_max_probability': float(test_probabilities.max(axis=1).mean()),
    'model_repository': TIMM_HF_REPO_ID,
    'model_revision': RESOLVED_REVISION,
    'backbone_documented_for_final_report': (
        'DINOv3 ViT-7B/16, timm repo timm/vit_7b_patch16_dinov3.lvd1689m, LVD-1689M pretraining checkpoint, '
        'revision {}, 100 percent frozen in FP16, sharded across two Tesla T4 GPUs. Only a 4.2M-parameter '
        'classifier head was trained on the official BDC Satria Data 2026 training images. No external '
        'labeled images and no test labels were used.'.format(RESOLVED_REVISION)
    ),
    'submission_path': str(SUBMISSION_PATH),
    'submission_validation': SUBMISSION_VALIDATION,
    'template_path': str(TEMPLATE_PATH),
    'test_dir': str(TEST_DIR),
    'test_order_sha256': TEST_ORDER_SHA256,
    'test_cache_signature': TEST_CACHE_MANIFEST['cache_signature'],
    'train_cache_signature': TRAIN_CACHE_MANIFEST[SELECTED_VARIANT]['cache_signature'],
    'checkpoint_reproduction': CHECKPOINT_REPRODUCTION,
}
write_json_atomic(TEST_INFERENCE_MANIFEST, ARTIFACT_DIR / 'test_inference_manifest.json')

REPRODUCIBILITY_MANIFEST = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'preflight_signature': PREFLIGHT_SIGNATURE,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
    'labels_sha256': LABELS_SHA256,
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SHA256,
    'test_order_sha256': TEST_ORDER_SHA256,
    'train_cache_signatures': {
        name: TRAIN_CACHE_MANIFEST[name]['cache_signature'] for name in VARIANT_ORDER
    },
    'test_cache_signature': TEST_CACHE_MANIFEST['cache_signature'],
    'model_revision': RESOLVED_REVISION,
    'environment': ENVIRONMENT_INFO,
    'gpus': GPU_RUNTIME_INFO,
    'total_runtime_minutes': elapsed_minutes(),
}
write_json_atomic(REPRODUCIBILITY_MANIFEST, ARTIFACT_DIR / 'reproducibility_manifest.json')

# ---------------------------------------------------------------------------
# THE ONE BLOCK TO COPY BACK
# ---------------------------------------------------------------------------
control_oof = OOF_METRICS[CONTROL_VARIANT]
native_oof = OOF_METRICS[NATIVE_VARIANT]

print()
print('#' * 78)
print('# V13 RESULT BLOCK  --  copy this whole block back')
print('#' * 78)
print('experiment                     : {}'.format(EXPERIMENT_NAME))
print('model revision                 : {}'.format(RESOLVED_REVISION))
print('fold sha256 matches V10        : True')
print('upsampled train rows           : {} of {}'.format(UPSAMPLED_ROWS, EXPECTED_TRAIN_ROWS))
print('pipeline integrity (control=V10): {}   delta {:+.10f}'.format(
    'PASS' if PIPELINE_INTEGRITY_OK else 'FAIL', CONTROL_REPRODUCTION['macro_f1_delta']
))
print('---- control  bilinear ---------------------------------------------------')
print('  OOF macro F1                 : {:.10f}'.format(control_oof['macro_f1']))
print('  OOF total errors             : {}'.format(control_oof['total_errors']))
print('  OOF R/O errors               : {}'.format(control_oof['ro_errors']))
print('  OOF Electronic F1            : {:.10f}'.format(control_oof['f1_electronic']))
print('  per fold macro F1            : {}'.format(
    {key: round(value, 10) for key, value in control_oof['per_fold_macro_f1'].items()}
))
print('---- treatment  bicubic (model-native) -----------------------------------')
print('  OOF macro F1                 : {:.10f}'.format(native_oof['macro_f1']))
print('  OOF total errors             : {}'.format(native_oof['total_errors']))
print('  OOF R/O errors               : {}'.format(native_oof['ro_errors']))
print('  OOF Electronic F1            : {:.10f}'.format(native_oof['f1_electronic']))
print('  per fold macro F1            : {}'.format(
    {key: round(value, 10) for key, value in native_oof['per_fold_macro_f1'].items()}
))
print('---- comparison ----------------------------------------------------------')
print('  OOF macro F1 gain            : {:+.10f}   (gate {:+.4f})'.format(
    OOF_MACRO_F1_GAIN, ADOPTION_MIN_OOF_MACRO_F1_GAIN
))
print('  fold 0 gain                  : {:+.10f}'.format(DEVELOPMENT_FOLD_GAIN))
print('  assessment folds won         : {} of 4'.format(FOLDS_WON))
print('  Electronic F1 drop           : {:+.10f}'.format(ELECTRONIC_F1_DROP))
print('  gain by seed                 : {}'.format({key: round(value, 8) for key, value in SEED_GAINS.items()}))
print('  control seed std             : {:.8f}'.format(CONTROL_SEED_SPREAD))
print('  gain exceeds 2x seed noise   : {}'.format(GAIN_EXCEEDS_SEED_NOISE))
print('---- gate ----------------------------------------------------------------')
for check_name, passed in ADOPTION_GATE_CHECKS.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
print('---- decision ------------------------------------------------------------')
print('  bicubic adopted              : {}'.format(NATIVE_ADOPTED))
print('  selected variant             : {}'.format(SELECTED_VARIANT))
print('  full training epochs         : {}'.format(FULL_TRAINING_EPOCHS))
print('  test class distribution      : {}'.format(PREDICTED_DISTRIBUTION))
print('  V10 test class distribution  : {}'.format(V10_REFERENCE['test_predicted_class_distribution']))
print('  reproduction max abs diff    : {:.3e}  (identical argmax: {})'.format(
    MAX_ABS_PROBABILITY_DIFFERENCE, PREDICTIONS_IDENTICAL
))
print('  submission                   : {}'.format(SUBMISSION_PATH))
print('  total runtime                : {:.0f} min'.format(elapsed_minutes()))
print('#' * 78)

## 7. Yang harus dibaca setelah notebook selesai

Sel terakhir mencetak **V13 RESULT BLOCK**. Salin blok itu apa adanya — itu satu-satunya hal yang perlu dikirim balik. Tidak perlu menafsirkan angkanya sendiri; keputusan adopsi/tolak sudah dihitung otomatis oleh gerbang terkunci.

### 7.1 Urutan pemeriksaan

**Langkah 1 — integritas pipeline.** Lihat `pipeline integrity (control=V10)`.

- `PASS` → lanjut ke langkah 2.
- `FAIL` → **berhenti menafsirkan**. Kontrol in-session gagal mereproduksi champion, jadi selisih bicubic-vs-bilinear tidak berarti apa-apa. Notebook otomatis menolak adopsi dan membangun submission dari resep kontrol (= resep V10). Laporkan `delta`-nya; kemungkinan penyebab tercatat di `control_reproduction.json` (revisi model berbeda, atau versi timm/torch berbeda).

**Langkah 2 — gerbang adopsi.** Blok `---- gate ----` mencetak sembilan baris `PASS`/`FAIL`. Adopsi terjadi hanya kalau **kesembilannya** `PASS`. Ini sudah otomatis; tidak ada keputusan manual.

**Langkah 3 — apakah gain-nya nyata atau derau.** Bandingkan dua angka:

- `OOF macro F1 gain`
- `control seed std` × 2

Kalau gain lebih kecil dari dua kali standar deviasi antar-seed, itu **derau**, walaupun kebetulan lolos ambang +0.0003. Baris `gain exceeds 2x seed noise` sudah menghitungnya. Gerbang formal tidak memblokir kasus ini — tapi jangan mengklaim kemenangan berdasarkan gain yang lebih kecil dari deraunya sendiri.

**Langkah 4 — apakah mekanismenya benar.** Hipotesisnya spesifik: bicubic membantu karena mempertahankan tekstur pada citra yang di-*upsample*, dan batas R/O adalah keputusan tekstur. Jadi:

- `OOF R/O errors` harus **turun**, dan
- `errors on upsampled` harus turun lebih banyak daripada `errors on native-res`.

Kalau Macro F1 naik tetapi error R/O tidak turun, mekanismenya salah — gain itu kebetulan, dan gerbang memang sudah menolaknya lewat syarat `oof_ro_errors_decrease`.

### 7.2 Empat kemungkinan hasil dan apa artinya

| Hasil | Arti | Tindakan |
|---|---|---|
| Semua gerbang `PASS`, gain > 2× derau, R/O turun | Champion baru. Kernel resize memang salah selama ini. | Submission dari bicubic sudah dibuat dan terverifikasi. Ini kandidat terkuat untuk submission terakhir. |
| Gain positif tapi < +0.0003, atau menang < 3 fold | Bicubic netral. Tidak ada bukti cukup. | Tolak. Submission dari kontrol = resep V10. Ablasi ini selesai dan tidak perlu diulang. |
| Gain negatif | Bilinear V7/V10 ternyata pilihan yang lebih baik untuk data ini, kemungkinan karena efek low-pass membantu pada citra kecil yang berisik. | Tolak. Ini hasil negatif yang berguna: hentikan seluruh cabang "geometri/resampling", termasuk V12B. |
| `pipeline integrity FAIL` | Bug atau perubahan lingkungan. | Laporkan delta-nya; jangan pakai hasil bicubic untuk keputusan apa pun. |

### 7.3 Yang **tidak** dilakukan notebook ini

- Tidak menyentuh data uji sebelum `selection_decision.json` ditulis.
- Tidak pernah membuat submission dari rata-rata fold, rata-rata seed, blending, TTA, atau flip. Persis satu head, satu forward pass, satu `argmax`.
- Tidak melakukan threshold tuning, kalibrasi, atau koreksi manual terhadap prediksi test.
- Tidak menghapus atau me-*relabel* satu pun baris train. Label validasi selalu label resmi asli.
- Tidak memakai dimensi citra sebagai fitur. Dimensi hanya dipakai untuk stratifikasi error (`errors on upsampled`) dan untuk sensus di §1.

### 7.4 Yang sengaja tidak dikerjakan di sini

Ketiganya adalah eksperimen terpisah, bukan bagian dari perubahan tunggal ini:

- **Geometri crop-pct timm** (resize sisi pendek 256 lalu center-crop). Itu mengubah *geometri* dan **membuang konten** di tepi; digabung dengan perubahan kernel akan menjadi dua perubahan sekaligus.
- **Aspect-preserving / letterbox 256.** Memerlukan masking patch padding pada `patch_mean`, jadi butuh dua perubahan agar valid.
- **Resolusi lebih tinggi (320/384).** 95% citra sudah di-*upsample* pada 256; menaikkan resolusi menambah token, bukan informasi. Mahal dan hipotesisnya lemah.

### 7.5 Artifact yang dihasilkan

```
/kaggle/working/v13_dinov3_vit7b_model_native_resampling/
├── submission_<TEAM_NAME>.csv        <- yang diunggah
├── final_head.pt                     <- checkpoint tunggal
├── selection_decision.json           <- ditulis SEBELUM test dibuka
├── control_reproduction.json         <- gerbang integritas pipeline
├── locked_contract.json              <- gerbang dikunci sebelum ada hasil
├── data_contract.json / fold_signature.json / fold_assignments.csv
├── preflight_report.json             <- termasuk bukti "interpolasi benar-benar aktif"
├── model_manifest.json               <- backbone untuk laporan akhir (juknis §5)
├── oof_metrics_<variant>.json, oof_predictions_<variant>.csv, oof_probabilities_<variant>.npy
├── per_fold_seed42.csv, seed_stability.csv
├── oof_errors_<selected>.csv, oof_recyclable_organic_errors_<selected>.csv
├── test_probabilities.npy, test_inference_manifest.json
├── checkpoint_reproduction_check.json, reproducibility_manifest.json
└── feature_cache/{bilinear_v10_control,bicubic_model_native}/{train,test}/
```

**Simpan `feature_cache/` sebagai Kaggle Dataset** kalau ada waktu. Kalau bicubic menang, cache itu adalah dasar semua eksperimen berikutnya dan tidak perlu 1,9 jam ekstraksi lagi.